# Trading Bot V6.3 · Live-Demo · Strategien A (Pullback) und B (Squeeze)

## Neu in V6.3: mehr Einstiege

V6.2 hat sehr wenige Trades gemacht. Die Strategien waren dabei nicht grundsätzlich zu streng. Einige Filter passen aber schlecht zu den Alpaca-Daten: Alpaca liefert nur die Kurse und das Volumen der **eigenen Handelsplattform**, und dort werden manche Coins dünn gehandelt. Dadurch fielen Coins stundenweise ganz heraus („zu wenig Handel“), Squeezes wurden verworfen, und die Volumenfilter bewerteten eher Zufall als echte Marktaktivität.

| Einstellung | V6.2 | V6.3 | Warum |
|---|---|---|---|
| A · Volumenfilter 15 Min | Volumen ≥ Durchschnitt | **aus** | Alpaca-Volumen = nur eigene Plattform, wenig aussagekräftig |
| B · Volumenfilter 15 Min | ≥ 1,5 × Durchschnitt | **≥ 1 × Durchschnitt** | wie oben, eine milde Bestätigung bleibt |
| A · Wartezeit auf 15-Min-Signal | 4 h | **8 h** | mehr Zeit, damit der Kurs nach dem Pullback wieder anzieht |
| Handelspause nach Verkauf | 24 h | **6 h** | ein neuer Einstieg braucht ohnehin einen neuen Pullback bzw. Squeeze |
| Datenprüfung stündlich | ≤ 5 von 50 h ohne Handel | **≤ 10 von 50 h** | Coins fielen sonst ganz aus der Analyse |
| Datenprüfung 15 Min | ≤ 2 von 20 Kerzen ohne Handel | **≤ 5 von 20** | wie oben |
| B · Stunden ohne Handel im Squeeze | keine | **höchstens 2 von 20** | eine einzelne ruhige Stunde verwarf sonst den ganzen Squeeze |
| Coins | 10 | **13** (+ AAVE, BCH, UNI) | mehr Gelegenheiten; fehlt ein Symbol bei Alpaca, wird der Coin weggelassen |

**Unverändert:** Marktlage (Trenddefinition), Pullback- und Squeeze-Erkennung, Einstieg über das Hoch der letzten 3 Kerzen bzw. über die Spanne, alle Stops und Ausstiege, 1 % Risiko pro Trade, 3 % Gesamtrisiko, höchstens 3 Positionen, 0,25 % Gebühr und 0,2 % Slippage.

**Neu im Cockpit: „Warum kein Trade?“** Das Cockpit zählt jetzt für jede Prüfstufe, woran ein möglicher Einstieg gescheitert ist: Datenprüfung, Pullback-Erkennung (A), Squeeze-Erkennung (B), die beiden 15-Minuten-Signale und das Risk Gate. So ist nach ein paar Tagen sichtbar, welche Bedingung die meisten Einstiege verhindert. Das ist die Grundlage für jede weitere Anpassung. Die Zählung liegt in `blocker_statistik_v63.json` und läuft über Neustarts weiter.

**Start-Hilfe.** Die Cockpit-Zelle zeigt jetzt immer einen blauen Knopf „Cockpit im Browser öffnen“ samt Link und versucht, den Tab selbst zu öffnen. Blockiert der Browser das, erscheint ein orangefarbener Hinweis. Die Startzelle prüft vor dem Start die Alpaca-Schlüssel: Fehlt ein Secret, ist es nicht für dieses Notebook freigegeben oder lehnt Alpaca die Schlüssel ab, steht dort und im Cockpit ein roter Hinweis mit der Lösung, und der Bot startet nicht.

**Eigenes Konto.** V6.3 startet mit einem frischen virtuellen Konto (`konto_v63_alpaca_usd.json`, 10.000 USD); das V6.2-Konto bleibt unangetastet.

**Wichtig:** Mehr Trades heißt nicht automatisch bessere Ergebnisse. Die Änderungen sind nicht mit historischen Daten getestet (Backtest). Die Auswertung je Strategie zeigt, ob die zusätzlichen Trades etwas taugen.

## Überblick

V6.3 läuft **live mit echten Alpaca-Kursen und handelt ausschließlich virtuell**. Es gibt **keine Alpaca-Orders**, im Code steht keine Order-Funktion, und ein Test prüft das. Gehandelt werden 13 Coins gegen USD (BTC, ETH, SOL, XRP, DOGE, ADA, AVAX, LINK, DOT, LTC, AAVE, BCH, UNI) mit einem virtuellen Budget von 10.000 USD. Das Konto liegt in `MyDrive/Trading_Bot/konto_v63_alpaca_usd.json`; die Konten von V4, V5 und V6 bleiben unangetastet.

## Takt: Wann analysiert der Bot?

| Wann | Was |
|---|---|
| **Bei jedem Start** und jede volle Stunde | Stundenkerzen holen, **Marktlage** je Coin (Aufwärtstrend / neutral / Abwärtstrend), Pullback (A) und Squeeze (B) erkennen, Ausstiege und Trailing-Stops offener Positionen |
| Alle 15 Minuten | 15-Minuten-Kerzen holen und für Coins mit bereitem Pullback bzw. aktivem Squeeze das **Einstiegssignal** prüfen, danach das Risk Gate |
| Alle 5 Minuten | Kurse (Mittelkurs aus Bid und Ask), Spread, Stops, Tagesverlustlimit |

Der Takt ist auf volle 5 Minuten plus 20 Sekunden ausgerichtet, damit die gerade abgeschlossene Kerze schon vorliegt. **Die Parameter verändert der Bot nie selbst.** „Anpassen“ heißt hier nur, dass die Marktlage je Coin bestimmt, welche Strategie gerade einsteigen darf.

## Strategie A · Momentum mit Pullback-Einstieg (Abschnitt 5)

1. **Aufwärtstrend** (4.1, vorläufig: EMA 50 über EMA 200 und ADX ≥ 20). Zusätzlich liegt der letzte Stundenschluss über dem EMA 50.
2. **Korrektur:** In den letzten 6 Stundenkerzen hat das Tief einer Kerze den EMA 20 von oben berührt oder unterschritten, die Kerze davor schloss über dem EMA 20. Der letzte Stundenschluss liegt wieder über dem EMA 20. *Ergänzt:* Kein Schlusskurs im Pullback liegt unter dem EMA 50.
3. **15-Minuten-Signal:** Der Schlusskurs liegt über dem höchsten Hoch der vorherigen 3 Kerzen, und der Aufwärtstrend ist weiter intakt. *V6.3:* kein Volumenfilter mehr. *Ergänzt:* Der Bot wartet höchstens 8 Stunden (V6.3, vorher 4) auf das Signal, und jeder Pullback wird nur einmal gehandelt.
- *Ergänzt, Stop und Ausstieg:* Der Stop liegt beim Pullback-Tief minus 0,5 × ATR. Danach gilt der 3 × ATR-Trailing-Stop. Schließt eine Stunde unter dem EMA 50, wird verkauft.

## Strategie B · Volatility Squeeze Breakout (Abschnitt 6)

1. **Kompression:** Die Bollinger-Bandbreite (20 Kerzen, 2 Standardabweichungen) liegt im niedrigsten 20-%-Bereich der **120 vorherigen** Bandbreiten; der aktuelle Wert zählt nicht zur eigenen Schwelle. Es darf kein Abwärtstrend vorliegen (4.3, vorläufig: EMA 50 unter EMA 200 und Schluss unter EMA 200). *Ergänzt:* Liegen in den 20 Kerzen mehr als 2 Stunden ohne Handel (V6.3, vorher: eine), zählt der Squeeze nicht, weil die flache Kerze die Bandbreite künstlich drückt.
2. **Spanne:** höchstes Hoch und tiefstes Tief der letzten 12 Stundenkerzen, festgehalten bei der ersten Squeeze-Stunde.
3. **15-Minuten-Signal:** Die Kerze schließt über der oberen Grenze, ihr Volumen liegt mindestens beim Durchschnitt der vorherigen 20 (V6.3, vorher 50 % darüber), und es besteht kein Abwärtstrend. Der Spread ist höchstens 0,15 %. Entschieden wird erst nach Kerzenschluss. *Ergänzt:* Liegt der Schluss mehr als 1 × ATR über der Spanne, wird nicht gekauft.
4. **Gültigkeit:** 12 Stunden. Jede Konsolidierungsphase wird nur einmal gehandelt; eine neue gibt es erst, wenn die Kompression zwischendurch geendet hat.
- *Ergänzt, Stop und Ausstieg:* Der Stop ist der engere von Spannenmitte und Einstieg minus 2 × ATR. Danach gilt der 3 × ATR-Trailing-Stop. Verkauft wird bei einem Stundenschluss zurück unter der oberen Grenze (Fehlausbruch) und nach 24 Stunden ohne Fortschritt von +1 × ATR.

## Gemeinsames Risikomanagement (Abschnitt 7, aus V4/V5)

- Höchstens 1 % Risiko pro Trade und 3 % Gesamtrisiko, jeweils bis zum Stop und inklusive Kosten.
- Höchstens 3 Positionen, höchstens 50 % des Kapitals je Position, eine Position pro Coin.
- Nach einem Verkauf gilt für den Coin eine Handelspause von 6 Stunden (V6.3, vorher 24).
- Das Tagesverlustlimit liegt bei 3 %.
- Die Kosten betragen 0,25 % Gebühr und 0,2 % Slippage je Seite.
- *Ergänzt:* Der Spread darf höchstens 0,15 % betragen. Treffen mehrere Signale zugleich ein, geht das mit dem geringsten Risiko pro Einheit zuerst.

**Vorläufig:** Die Definitionen aus 4.1 und 4.3 sowie alle mit *Ergänzt* markierten Punkte sind Annahmen, bis die fehlenden Abschnitte vorliegen. Alle Werte stehen als Einstellungen in Abschnitt 3 des Bot-Moduls und werden im Cockpit als vorläufig angezeigt.

## Einrichtung und Cockpit

In Colab unter 🔑 **Secrets** `ALPACA_PAPER_API_KEY` und `ALPACA_PAPER_SECRET_KEY` freigeben, dann „Alle ausführen“. Das Cockpit hat eine eigene Zelle, einen Browser-Tab, der sich beim Start selbst öffnet (der Link bleibt als Rückfall), und die Datei `MyDrive/Trading_Bot/cockpit_v63_latest.html`. Es zeigt je Coin die Marktlage, den Status von Pullback und Squeeze mit Spanne und Ablaufzeit sowie die letzte 15-Minuten-Prüfung mit Begründung. Bei den offenen Positionen steht die jeweilige Strategie, darunter die Auswertung je Strategie und die Statistik „Warum kein Trade?“.

Ergebnisse im Paper Trading sind Modellwerte und kein Rentabilitätsnachweis.

In [ ]:
%pip -q install alpaca-py

In [ ]:
%%writefile /content/crypto_bot_v63.py
"""Trading Bot V6.3: Strategien A (Pullback) und B (Squeeze), Alpaca-Live-Daten, virtuell. Keine Orders."""
# ============================================================
#
# TRADING BOT V6.3 · LIVE-DEMO
#
# ALPACA-LIVE-DATEN (nur lesend) + VIRTUELLES PAPER TRADING
# STRATEGIE A: Momentum mit Pullback-Einstieg (1h + 15 Min)
# STRATEGIE B: Volatility Squeeze Breakout (1h + 15 Min)
# Gemeinsames Risikomanagement aus V4/V5 (1 % / 3 %, max. 3 Positionen)
#
# 13 Coins gegen USD · virtuelles Budget 10.000 USD
#
# Takt: alle 5 Minuten Kurse und Stops, alle 15 Minuten
# Einstiegssignale, jede volle Stunde (und bei jedem Start)
# Marktlage, Pullback-/Squeeze-Erkennung und Ausstiege.
#
# Die Parameter sind vorläufige Testparameter und werden vom
# Bot nie selbst verändert (siehe Abschnitt 3).
#
# Das Modul hat beim Import keine Seiteneffekte (kein Drive,
# keine Alpaca-Abfrage) - gestartet wird über start().
#
# ============================================================


# ============================================================
# 1. BIBLIOTHEKEN
# ============================================================

import requests
import pandas as pd

from alpaca.common.exceptions import APIError
from alpaca.data.requests import (
    CryptoBarsRequest,
    CryptoLatestQuoteRequest
)
from alpaca.data.timeframe import TimeFrame, TimeFrameUnit

import json
import time
import math
import os
import re
import shutil
import traceback

from collections import deque
from pathlib import Path
from datetime import datetime, timedelta, timezone
from zoneinfo import ZoneInfo


# ------------------------------------------------------------
# AUSGABE-DESIGN (Farben, Kästen, Emojis für Google Colab)
# ------------------------------------------------------------

FARBE_GRUEN = "\033[92m"
FARBE_ROT = "\033[91m"
FARBE_GELB = "\033[93m"
FARBE_BLAU = "\033[94m"
FARBE_CYAN = "\033[96m"
FARBE_GRAU = "\033[90m"
FARBE_FETT = "\033[1m"
FARBE_ENDE = "\033[0m"

BREITE = 60


def faerben(text, farbe):

    return f"{farbe}{text}{FARBE_ENDE}"


def wert_faerben(zahl, text):

    # Grün bei >= 0, Rot bei < 0 - für Gewinn/Verlust-Angaben

    return faerben(
        text,
        FARBE_GRUEN if zahl >= 0 else FARBE_ROT
    )


def kasten(titel, emoji=""):

    linie = "═" * BREITE

    print("")

    print(faerben(linie, FARBE_BLAU))

    kopf = f"{emoji}  {titel}" if emoji else titel

    print(faerben(kopf.center(BREITE), FARBE_FETT))

    print(faerben(linie, FARBE_BLAU))


def unterkasten(titel, emoji=""):

    kopf = f"{emoji}  {titel}" if emoji else titel

    print("")

    print(faerben(kopf, FARBE_FETT))

    print(faerben("─" * BREITE, FARBE_GRAU))


def trenner():

    print(faerben("─" * BREITE, FARBE_GRAU))


def zeile(label, wert, emoji=None, breite_label=26):

    praefix = f"{emoji} " if emoji else ""

    print(
        f"{praefix}{label:<{breite_label}} {wert}"
    )


# --- Dashboard-Boxen (┌─┐ │ └─┘) für Portfolio/Positionen/Signale ---

INNEN_BREITE = 62

# Emojis/Sonderzeichen wirken in Monospace-Schriften (Colab)
# doppelt so breit wie ihre Länge in Python (len() zählt 1
# Zeichen). Für einigermaßen saubere Rahmen wird das hier grob
# nachgebildet - perfekte Pixel-Ausrichtung ist nicht das Ziel.
# ANSI-Farbcodes sind unsichtbar und zählen nicht zur Breite.

ANSI_MUSTER = re.compile(r"\033\[[0-9;]*m")


def sichtbare_breite(text):

    text_ohne_farbcodes = ANSI_MUSTER.sub("", text)

    breite = 0

    for zeichen in text_ohne_farbcodes:

        code = ord(zeichen)

        if code == 0xFE0F:

            continue

        if (
            0x1F300 <= code <= 0x1FAFF
            or 0x2600 <= code <= 0x27BF
            or code in (0x25B2, 0x25BC, 0x25CF)
        ):

            breite += 2

        else:

            breite += 1

    return breite


def box_start(titel, emoji=""):

    kopf_text = f"{emoji} {titel}" if emoji else titel

    kopf = f"┌─ {kopf_text} "

    # Gesamtbreite der oberen Rahmenlinie muss der Breite von
    # box_zeile()/box_ende() entsprechen: "│ " + INNEN_BREITE +
    # " │" bzw. "└" + "─"*(INNEN_BREITE+2) + "┘" = INNEN_BREITE+4

    rest = (
        INNEN_BREITE + 4
        - sichtbare_breite(kopf)
        - 1
    )

    if rest < 0:

        rest = 0

    print("")

    print(
        faerben(kopf, FARBE_BLAU)
        + faerben("─" * rest + "┐", FARBE_BLAU)
    )


def box_zeile(text=""):

    innen = sichtbare_breite(text)

    fuellung = max(
        0,
        INNEN_BREITE - innen
    )

    print(
        faerben("│ ", FARBE_BLAU)
        + text
        + " " * fuellung
        + faerben(" │", FARBE_BLAU)
    )


def spalte(text, breite):

    # ANSI-sicheres Auffüllen einer Tabellenspalte auf feste
    # sichtbare Breite - normales str.ljust() würde die
    # unsichtbaren Farbcodes mitzählen und die Spalte dadurch
    # zu schmal auffüllen.

    fehlend = max(
        0,
        breite - sichtbare_breite(text)
    )

    return text + " " * fehlend


def box_ende():

    print(
        faerben(
            "└" + "─" * (INNEN_BREITE + 2) + "┘",
            FARBE_BLAU
        )
    )


def balken(anteil, breite=20):

    anteil = max(0.0, min(1.0, anteil))

    gefuellt = round(anteil * breite)

    farbe = FARBE_GRUEN if anteil < 0.8 else FARBE_ROT

    fuellstrich = faerben("█" * gefuellt, farbe)

    leerstrich = faerben(
        "░" * (breite - gefuellt),
        FARBE_GRAU
    )

    return f"[{fuellstrich}{leerstrich}]"


def pfeil(wert):

    if wert > 0:

        return faerben("▲", FARBE_GRUEN)

    if wert < 0:

        return faerben("▼", FARBE_ROT)

    return faerben("─", FARBE_GRAU)


def signal_text(signal):

    if signal == "LONG":

        return faerben("🟢 LONG", FARBE_GRUEN)

    return faerben("⚪ NO TRADE", FARBE_GRAU)


VERSION = "6.3"


# ============================================================
# 2. GOOGLE DRIVE VERBINDEN
# ============================================================

# Passiert erst in start() - siehe Abschnitt 27.


# ============================================================
# 3. GRUNDEINSTELLUNGEN
# ============================================================

STARTKAPITAL = 10000.00

# Ausschließlich Paper Trading
PAPER_TRADING = True

# Automatische virtuelle Käufe
AUTOMATISCHE_KAEUFE = True

# Takt: Kursüberwachung alle 5 Minuten, ausgerichtet auf volle
# 5 Minuten + Verzögerung, damit Alpaca die gerade
# abgeschlossene 15-Minuten- bzw. Stundenkerze schon liefert
INTERVALL = 300
TAKT_VERZOEGERUNG = 20

# Risikomanagement
MAX_RISIKO_PRO_TRADE = 0.01
MAX_GESAMTRISIKO = 0.03
MAX_POSITIONSANTEIL = 0.50

MAX_OFFENE_POSITIONEN = 3

MAX_TAGESVERLUST = 0.03

# Handelsstrategie: Trendfolge mit ATR-Positionsgröße
# und ATR-Trailing-Stop (kein fixes Take-Profit, siehe Analyse
# zur Kostenasymmetrie: fixe kleine Targets verlieren gegen
# Gebühren+Slippage, laufengelassene Trends tragen sie)

TREND_FILTER_PERIODE = 200      # langsamer Trendfilter (MA200)

ADX_PERIODE = 14
ADX_MINDESTWERT = 20            # Mindest-Trendstärke für ein Signal

ATR_PERIODE = 14
ATR_STOP_MULTIPLIKATOR = 2.0        # initialer Stop = Einstieg - 2x ATR
ATR_TRAILING_MULTIPLIKATOR = 3.0    # Trailing-Stop = höchster Schluss seit Einstieg - 3x ATR

# Mindestanzahl an Stundenkerzen, bevor gehandelt wird
# (Platz für EMA200 + Warmlaufzeit von ATR/ADX)
MIN_ERFORDERLICHE_KERZEN = TREND_FILTER_PERIODE + 50

# ------------------------------------------------------------
# STRATEGIEN A UND B - vorläufige Testparameter
# Nicht anhand einzelner Testergebnisse verändern; Änderungen
# nur von Hand, mit neuer Version und nach Freigabe. Der Bot
# passt keinen dieser Werte selbst an.
# ------------------------------------------------------------

EMA_KURZ = 20
EMA_MITTEL = 50
EMA_LANG = 200

# VORLÄUFIG, bis Abschnitt 4.1/4.3 vorliegen:
# Aufwärtstrend (4.1) = EMA50 > EMA200 und ADX14 >= 20
# Abwärtstrend  (4.3) = EMA50 < EMA200 und Schluss < EMA200
TREND_DEFINITION_VORLAEUFIG = True

# Strategie A - Momentum mit Pullback-Einstieg
PULLBACK_KERZEN = 6             # Prüfung 2: letzte 6 Stundenkerzen
PULLBACK_GUELTIG_STUNDEN = 8    # V6.3: 8 statt 4 h auf das 15-Min-Signal warten
A_AUSBRUCH_KERZEN = 3           # Prüfung 3: Schluss über Hoch der vorherigen 3
# V6.3: 0 = kein Volumenfilter für A. Alpaca liefert nur das Volumen
# der eigenen Handelsplattform - dünn und für die Marktbewegung
# wenig aussagekräftig; der Filter hat vor allem Signale verworfen.
A_VOLUMEN_FAKTOR = 0.0
A_STOP_ATR = 0.5                # Ergänzung: Stop = Pullback-Tief - 0,5 x ATR

# Strategie B - Volatility Squeeze Breakout
BB_PERIODE = 20
BB_STDABW = 2.0
SQUEEZE_REFERENZ = 120          # Vergleichszeitraum der Bandbreite
SQUEEZE_QUANTIL = 0.20          # niedrigste 20 %
SPANNE_KERZEN = 12              # Handelsspanne: letzte 12 Stundenkerzen
SQUEEZE_GUELTIG_STUNDEN = 12
B_VOLUMEN_FAKTOR = 1.0          # V6.3: Volumen >= Durchschnitt der vorherigen 20 (vorher 1,5 x)
B_UEBERDEHNUNG_ATR = 1.0        # Ergänzung: Schluss höchstens 1 x ATR über Spanne
B_STOP_ATR = 2.0                # Ergänzung: Stop = engerer von Spannenmitte / 2 x ATR
B_ZEITSTOP_STUNDEN = 24         # Ergänzung: nach 24 h ohne +1 x ATR verkaufen
B_MAX_STUNDEN_OHNE_HANDEL = 2   # V6.3: höchstens 2 der 20 Bandbreiten-Stunden ohne Trade (vorher 0)

# Gemeinsam
VOLUMEN_VERGLEICH_KERZEN = 20
MAX_15M_OHNE_HANDEL = 5         # V6.3: höchstens 5 der 20 Vergleichskerzen ohne Trade (vorher 2)
MAX_SPREAD = 0.0015             # Ergänzung: Spread höchstens 0,15 %

STRATEGIE_NAMEN = {
    "A": "A · Pullback",
    "B": "B · Squeeze"
}

# Simulierte Handelskosten je Seite. V6.2: Gebühr 0,25 % statt
# 0,1 % - das entspricht der Alpaca-Krypto-Taker-Gebühr in der
# niedrigsten Stufe, denn der Bot handelt wie mit Market-Orders.
# Slippage unverändert 0,2 %.
GEBUEHR = 0.0025
SLIPPAGE = 0.002

# Handelspause nach Verkauf. V6.3: 6 statt 24 Stunden - ein neuer
# Einstieg braucht ohnehin einen neuen Pullback bzw. Squeeze.
HANDELSPAUSE = 6 * 3600

# Alpaca-Daten
ERSTER_ABRUF_STUNDEN = 720      # erster Abruf: 30 Tage Stundenkerzen
MAX_QUOTE_ALTER = 300           # Quote älter als 5 Min. = kein Handel
MAX_STUNDEN_OHNE_HANDEL = 10    # V6.3: höchstens 10 der letzten 50 Stunden (vorher 5)
MAX_LUECKE_AM_STUECK = 3        # höchstens 3 Stunden am Stück ohne Handel

# Sekunden pro Stundenkerze
KERZEN_SEKUNDEN = 3600


# ============================================================
# 4. KRYPTOWÄHRUNGEN
# ============================================================

# Coins ohne gültigen Alpaca-Kurs beim Start werden für diesen
# Lauf weggelassen (siehe handelspaare_ermitteln).

COINS = {

    "bitcoin": {
        "name": "Bitcoin",
        "symbol": "BTC/USD"
    },

    "ethereum": {
        "name": "Ethereum",
        "symbol": "ETH/USD"
    },

    "solana": {
        "name": "Solana",
        "symbol": "SOL/USD"
    },

    "xrp": {
        "name": "XRP",
        "symbol": "XRP/USD"
    },

    "dogecoin": {
        "name": "Dogecoin",
        "symbol": "DOGE/USD"
    },

    "cardano": {
        "name": "Cardano",
        "symbol": "ADA/USD"
    },

    "avalanche": {
        "name": "Avalanche",
        "symbol": "AVAX/USD"
    },

    "chainlink": {
        "name": "Chainlink",
        "symbol": "LINK/USD"
    },

    "polkadot": {
        "name": "Polkadot",
        "symbol": "DOT/USD"
    },

    "litecoin": {
        "name": "Litecoin",
        "symbol": "LTC/USD"
    },

    # V6.3: drei weitere Coins für mehr Gelegenheiten. Fehlt ein
    # Symbol bei Alpaca, wird der Coin beim Start weggelassen.

    "aave": {
        "name": "Aave",
        "symbol": "AAVE/USD"
    },

    "bitcoin_cash": {
        "name": "Bitcoin Cash",
        "symbol": "BCH/USD"
    },

    "uniswap": {
        "name": "Uniswap",
        "symbol": "UNI/USD"
    }

}


# ============================================================
# 5. SPEICHERORTE
# ============================================================

STANDARD_ORDNER = Path(
    "/content/drive/MyDrive/Trading_Bot"
)

# Werden von speicherorte_einrichten() gesetzt (in Tests auf
# einen temporären Ordner statt Google Drive).

ORDNER = None
MARKTDATEN_ORDNER = None
KONTO_DATEI = None
BACKUP_DATEI = None
ANALYSE_DATEI = None
STRATEGIE_DATEI = None
STATISTIK_DATEI = None


def speicherorte_einrichten(ordner=STANDARD_ORDNER):

    global ORDNER, MARKTDATEN_ORDNER, KONTO_DATEI
    global BACKUP_DATEI, ANALYSE_DATEI, STRATEGIE_DATEI, STATISTIK_DATEI

    ORDNER = Path(ordner)

    ORDNER.mkdir(
        parents=True,
        exist_ok=True
    )

    MARKTDATEN_ORDNER = ORDNER / "Marktdaten_Alpaca_v6"

    MARKTDATEN_ORDNER.mkdir(
        parents=True,
        exist_ok=True
    )

    # Eigene Kontodatei für V6.3: Mit den gelockerten Filtern sind
    # die Ergebnisse nicht mit V6.2 vergleichbar, die Auswertung je
    # Strategie startet deshalb mit einem frischen Konto. Die Konten
    # von V4, V5, V6 und V6.2 bleiben unangetastet.

    KONTO_DATEI = ORDNER / "konto_v63_alpaca_usd.json"

    # Sicherung der bisherigen Kontodatei

    BACKUP_DATEI = ORDNER / "konto_v63_alpaca_usd_backup.json"

    # Zustand der Strategien A/B je Coin (bereiter Pullback,
    # aktiver Squeeze mit Spanne) - überlebt einen Neustart

    STRATEGIE_DATEI = ORDNER / "strategie_zustand_v63.json"

    # Ergebnis der letzten stündlichen Marktanalyse - nur für
    # die Anzeige, damit das Cockpit nach einem Neustart
    # innerhalb derselben Stunde nicht leer ist.

    ANALYSE_DATEI = ORDNER / "marktlage_v63.json"

    # V6.3: wie oft welche Prüfung einen Einstieg verhindert hat

    STATISTIK_DATEI = ORDNER / "blocker_statistik_v63.json"


# ============================================================
# 6. HILFSFUNKTIONEN
# ============================================================

def zeitstempel():

    # Intern wird konsequent UTC gespeichert (Kontodatei,
    # Transaktionen, Alpaca-Stundenkerzen) - das hält Tages-
    # limit-Reset, Handelspause und Kerzengrenzen robust
    # gegen die Zeitumstellung. Nur die Anzeige für den
    # Menschen läuft über lokale_zeit_anzeigen() unten.

    return datetime.now(
        timezone.utc
    ).isoformat()


BERLIN_ZEITZONE = ZoneInfo("Europe/Berlin")


def lokale_zeit_anzeigen(zeitpunkt_utc):

    # Wandelt einen UTC-Zeitpunkt für die Anzeige nach
    # Europe/Berlin um - mit dem korrekten deutschen Kürzel
    # (MESZ im Sommer, MEZ im Winter). datetime.strftime("%Z")
    # liefert dafür nur "CEST"/"CET" (englisch), daher hier
    # manuell über die Sommerzeit-Erkennung gesetzt.

    lokal = zeitpunkt_utc.astimezone(BERLIN_ZEITZONE)

    ist_sommerzeit = (
        lokal.dst() is not None
        and lokal.dst().total_seconds() > 0
    )

    kuerzel = "MESZ" if ist_sommerzeit else "MEZ"

    return lokal.strftime(f"%d.%m.%Y %H:%M:%S {kuerzel}")


def usd(wert):

    # Kleine Kurse (DOGE, XRP, ADA ...) brauchen mehr
    # Nachkommastellen, sonst sind Stop und ATR nicht lesbar.

    betrag = abs(wert)

    stellen = (
        2 if betrag >= 1 or betrag == 0
        else 4 if betrag >= 0.01
        else 6
    )

    text = f"{wert:,.{stellen}f}"

    text = (
        text.replace(",", "X")
        .replace(".", ",")
        .replace("X", ".")
    )

    return text + " USD"


def gueltige_zahl(wert):

    return (
        isinstance(wert, (int, float))
        and not isinstance(wert, bool)
        and math.isfinite(wert)
    )


# Ereignisprotokoll für das Cockpit (nur im Speicher, geht bei
# einem Neustart verloren - Käufe/Verkäufe stehen dauerhaft in
# den Transaktionen der Kontodatei).

EREIGNISSE = deque(maxlen=50)


def ereignis(art, coin=None, text=""):

    EREIGNISSE.append({
        "zeit": zeitstempel(),
        "art": art,
        "coin": coin,
        "text": text
    })


# ============================================================
# 7. ALPACA-MARKTDATEN (nur lesend)
# ============================================================

# V5 nutzt Alpaca ausschließlich als Datenquelle - wie der
# Aktien-Bot. Es gibt in diesem Modul keine Order-Funktion;
# das Alpaca-Paper-Konto wird beim Start nur einmal gelesen
# (konto_snapshot_lesen) und nie mit dem virtuellen Konto
# verrechnet.

# Werden in start() gesetzt: CryptoHistoricalDataClient und
# der einmalige Start-Snapshot des Alpaca-Paper-Kontos
DATENQUELLE = None
ALPACA_KONTO = None


class DatenFehler(RuntimeError):

    # Vorübergehend fehlende oder unbrauchbare Marktdaten:
    # kein Handel in diesem Durchlauf, der Bot läuft weiter.

    pass


# Fehler, bei denen nur dieser Durchlauf ausfällt (statt den
# Bot aus Sicherheitsgründen zu stoppen)

DATEN_FEHLERTYPEN = (
    requests.exceptions.RequestException,
    APIError,
    DatenFehler
)


def datenquelle_pruefen():

    if DATENQUELLE is None:

        raise RuntimeError(
            "Keine Alpaca-Datenquelle gesetzt (start() aufrufen)."
        )

    return DATENQUELLE


def konto_snapshot_lesen(broker):

    # Einmaliger, rein lesender Blick auf das Alpaca-Paper-
    # Konto (wie der Start-Snapshot des Aktien-Bots). Nur zur
    # Anzeige; der Bot rechnet mit seinem eigenen Budget.

    konto = broker.get_account()

    return {
        "equity": float(konto.equity),
        "waehrung": str(getattr(konto, "currency", "USD")),
        "zeit": zeitstempel()
    }


# ============================================================
# 8. SYMBOLE PRÜFEN
# ============================================================

# Wird in start() über handelspaare_ermitteln() gesetzt:
# coin -> Alpaca-Symbol, nur Coins mit gültigem Kurs beim Start
HANDELSPAARE = {}


def handelspaare_ermitteln():

    # Prüft einmalig, dass Alpaca für alle Symbole einen
    # frischen Kurs liefert. Ergebnis: coin -> Alpaca-Symbol.

    print("")
    print("Prüfe Alpaca-Krypto-Symbole ...")

    alle = {
        coin: info["symbol"]
        for coin, info in COINS.items()
    }

    kurse, fehlende = kurse_abrufen(alle)

    for coin, symbol in alle.items():

        if coin in kurse:

            print(
                f"{COINS[coin]['name']}: {symbol} "
                f"({usd(kurse[coin])})"
            )

        else:

            print(
                faerben(
                    f"{COINS[coin]['name']}: wird weggelassen - "
                    f"{fehlende[coin]}",
                    FARBE_GELB
                )
            )

    if not kurse:

        raise DatenFehler(
            "Alpaca liefert für keinen Coin einen Kurs."
        )

    return {
        coin: symbol
        for coin, symbol in alle.items()
        if coin in kurse
    }


# ============================================================
# 9. AKTUELLE KURSE ABRUFEN
# ============================================================

def kurse_abrufen(symbole):

    # Mittelkurs aus Bid und Ask des letzten Alpaca-Quotes.
    # Quotes sind auch in handelsarmen Minuten aktuell, der
    # letzte Trade dagegen kann bei SOL/USD älter sein.

    # Gibt (kurse, fehlende) zurück: ein unbrauchbarer Quote
    # betrifft nur seinen Coin, nicht die übrigen.

    antwort = datenquelle_pruefen().get_crypto_latest_quote(
        CryptoLatestQuoteRequest(
            symbol_or_symbols=list(symbole.values())
        )
    )

    jetzt = datetime.now(timezone.utc)

    kurse = {}

    fehlende = {}

    for coin, symbol in symbole.items():

        try:

            kurse[coin], SPREADS[coin] = quote_mittelkurs(
                antwort.get(symbol),
                symbol,
                jetzt
            )

        except DatenFehler as e:

            fehlende[coin] = str(e)

            SPREADS.pop(coin, None)

    return kurse, fehlende


# Relativer Spread (Ask - Bid) / Mittelkurs der letzten
# Kursprüfung je Coin - für die Spread-Prüfung im Risk Gate
SPREADS = {}


def quote_mittelkurs(quote, symbol, jetzt):

    if quote is None:

        raise DatenFehler(
            f"Kein Alpaca-Quote: {symbol}"
        )

    bid = float(quote.bid_price)

    ask = float(quote.ask_price)

    if (
        not gueltige_zahl(bid)
        or not gueltige_zahl(ask)
        or bid <= 0
        or ask < bid
    ):

        raise DatenFehler(
            f"Ungültiger Alpaca-Quote: {symbol}"
        )

    zeitpunkt = quote.timestamp

    if zeitpunkt.tzinfo is None:

        zeitpunkt = zeitpunkt.replace(tzinfo=timezone.utc)

    alter = (jetzt - zeitpunkt).total_seconds()

    if alter > MAX_QUOTE_ALTER:

        raise DatenFehler(
            f"Alpaca-Quote veraltet ({alter / 60:.0f} Min.): "
            f"{symbol}"
        )

    mitte = (bid + ask) / 2

    return mitte, (ask - bid) / mitte


# Coins, deren Kurs in der letzten Kursprüfung fehlte
# (nur Anzeige) - coin -> Grund

KURSFEHLER = {}


def aktuelle_kurse(konto=None):

    kurse, fehlende = kurse_abrufen(HANDELSPAARE)

    KURSFEHLER.clear()

    KURSFEHLER.update(fehlende)

    for coin, grund in fehlende.items():

        print(faerben(f"⚠️  {grund}", FARBE_GELB))

    # Ohne Kurs für eine offene Position lassen sich weder
    # Stop noch Kontowert prüfen: dann kein Handel in diesem
    # Durchlauf (die übrigen Coins sind egal).

    offene = konto["positionen"] if konto is not None else {}

    fehlend_offen = [coin for coin in offene if coin not in kurse]

    if fehlend_offen:

        raise DatenFehler(
            "Kein Kurs für offene Position: "
            + ", ".join(fehlende[coin] for coin in fehlend_offen)
        )

    return kurse


# ============================================================
# 10. HISTORISCHE KURSE SPEICHERN
# ============================================================

# Zwei Zeiträume: Stundenkerzen (Marktlage, Squeeze, Pullback)
# und 15-Minuten-Kerzen (Einstiegssignale)

ZEITRAEUME = {
    "1h": {
        "freq": "h",
        "alpaca": TimeFrame.Hour,
        "erster_abruf": pd.Timedelta(hours=720)
    },
    "15m": {
        "freq": "15min",
        "alpaca": TimeFrame(15, TimeFrameUnit.Minute),
        "erster_abruf": pd.Timedelta(days=3)
    }
}


def kursdatei(coin, zeitraum="1h"):

    return (
        MARKTDATEN_ORDNER
        / f"{coin}_{zeitraum}.csv"
    )


def kurse_laden(coin, zeitraum="1h"):

    datei = kursdatei(coin, zeitraum)

    if not datei.exists():

        return pd.DataFrame(
            columns=[
                "open", "high", "low", "close",
                "volume", "ohne_handel"
            ],
            index=pd.DatetimeIndex([], tz="UTC", name="zeit")
        )

    df = pd.read_csv(
        datei,
        parse_dates=["zeit"],
        index_col="zeit"
    )

    df.index = pd.to_datetime(
        df.index,
        utc=True
    )

    if df.index.has_duplicates:

        raise ValueError(
            f"Doppelte Zeitstempel in {datei}"
        )

    df["ohne_handel"] = df["ohne_handel"].astype(bool)

    df = df.sort_index()

    return df


def kurse_speichern(coin, df, zeitraum="1h"):

    datei = kursdatei(coin, zeitraum)

    temporaer = Path(
        str(datei) + ".tmp"
    )

    df.to_csv(
        temporaer,
        index_label="zeit"
    )

    os.replace(
        temporaer,
        datei
    )


# ============================================================
# 11. HISTORISCHE KURSE AKTUALISIEREN
# ============================================================

def stunden_ohne_handel_auffuellen(df, bis, freq="h"):

    # Alpaca liefert für eine Stunde ohne Trade keine Kerze.
    # Solche Stunden werden als flache Kerze auf dem letzten
    # Schlusskurs eingetragen (Volumen 0, ohne_handel=True),
    # damit MA/ATR/ADX auf einem lückenlosen Stundenraster
    # rechnen. marktdaten_pruefen() begrenzt, wie viele davon
    # in den letzten 50 Stunden erlaubt sind.

    if df.empty:

        return df

    raster = pd.date_range(
        df.index[0],
        bis,
        freq=freq,
        tz="UTC",
        name="zeit"
    )

    df = df.reindex(raster)

    fehlend = df["close"].isna()

    df["close"] = df["close"].ffill()

    for spalte_name in ["open", "high", "low"]:

        df.loc[fehlend, spalte_name] = df.loc[fehlend, "close"]

    df.loc[fehlend, "volume"] = 0.0

    df["ohne_handel"] = (
        df["ohne_handel"].astype("boolean").fillna(True).astype(bool)
    )

    return df


def historische_kurse(coin, zeitraum="1h"):

    # Für "15m" gilt alles sinngemäß mit Viertelstunden statt
    # Stunden (die Variablennamen stammen aus V5).

    art = ZEITRAEUME[zeitraum]

    periode = pd.Timedelta(art["freq"] if art["freq"] != "h" else "1h")

    df_alt = kurse_laden(coin, zeitraum)

    aktuelle_stunde = (
        pd.Timestamp.now(tz="UTC").floor(art["freq"])
    )

    # Bereits gespeicherte Daten nicht erneut vollständig
    # herunterladen; die letzte bekannte Kerze nochmals
    # abrufen (spät eingetroffene Trades, aufgefüllte Stunden).

    if df_alt.empty:

        start = aktuelle_stunde - art["erster_abruf"]

    else:

        start = df_alt.index[-1] - periode

    symbol = HANDELSPAARE[coin]

    antwort = datenquelle_pruefen().get_crypto_bars(
        CryptoBarsRequest(
            symbol_or_symbols=symbol,
            timeframe=art["alpaca"],
            start=start.to_pydatetime()
        )
    )

    kerzen = antwort.data.get(symbol, [])

    zeilen = []

    for kerze in kerzen:

        zeit = pd.Timestamp(kerze.timestamp)

        zeit = (
            zeit.tz_localize("UTC") if zeit.tzinfo is None
            else zeit.tz_convert("UTC")
        )

        # Noch laufende Stundenkerze ausschließen

        if zeit >= aktuelle_stunde:

            continue

        zeilen.append({

            "zeit": zeit,

            "open": float(kerze.open),

            "high": float(kerze.high),

            "low": float(kerze.low),

            "close": float(kerze.close),

            "volume": float(kerze.volume),

            "ohne_handel": False

        })

    if not zeilen and df_alt.empty:

        raise DatenFehler(
            f"Keine abgeschlossenen Kerzen: {coin}"
        )

    df_neu = pd.DataFrame(
        zeilen,
        columns=[
            "zeit", "open", "high", "low", "close",
            "volume", "ohne_handel"
        ]
    ).set_index("zeit")

    # Bereits gespeicherte und neue Kurse zusammenführen;
    # eine echte Kerze ersetzt eine früher aufgefüllte.

    teile = [teil for teil in (df_alt, df_neu) if not teil.empty]

    df = pd.concat(teile) if len(teile) > 1 else teile[0]

    df = df[
        ~df.index.duplicated(keep="last")
    ]

    df = df.sort_index()

    # Nur vollständige Stunden speichern

    df = df[
        df.index < aktuelle_stunde
    ]

    df = stunden_ohne_handel_auffuellen(
        df,
        aktuelle_stunde - periode,
        art["freq"]
    )

    # Daten plausibilisieren

    for spalte_name in [
        "open",
        "high",
        "low",
        "close"
    ]:

        if (
            df[spalte_name].isna().any()
            or (df[spalte_name] <= 0).any()
        ):

            raise ValueError(
                f"Ungültige historische Kurse: {coin}"
            )

    # Neue Daten dauerhaft speichern

    kurse_speichern(
        coin,
        df,
        zeitraum
    )

    return df


# ============================================================
# 12. TECHNISCHE INDIKATOREN (ATR, ADX)
# ============================================================

def true_range_berechnen(df):

    vorheriger_schluss = df["close"].shift(1)

    bereich = pd.concat([

        df["high"] - df["low"],

        (df["high"] - vorheriger_schluss).abs(),

        (df["low"] - vorheriger_schluss).abs()

    ], axis=1).max(axis=1)

    return bereich


def atr_berechnen(df, periode=ATR_PERIODE):

    tr = true_range_berechnen(df)

    atr = tr.ewm(
        alpha=1 / periode,
        adjust=False,
        min_periods=periode
    ).mean()

    return atr


def adx_berechnen(df, periode=ADX_PERIODE):

    hoch_diff = df["high"].diff()

    tief_diff = -df["low"].diff()

    plus_dm = hoch_diff.where(
        (hoch_diff > tief_diff) & (hoch_diff > 0),
        0.0
    )

    minus_dm = tief_diff.where(
        (tief_diff > hoch_diff) & (tief_diff > 0),
        0.0
    )

    tr = true_range_berechnen(df)

    tr_geglaettet = tr.ewm(
        alpha=1 / periode,
        adjust=False,
        min_periods=periode
    ).mean()

    plus_dm_geglaettet = plus_dm.ewm(
        alpha=1 / periode,
        adjust=False,
        min_periods=periode
    ).mean()

    minus_dm_geglaettet = minus_dm.ewm(
        alpha=1 / periode,
        adjust=False,
        min_periods=periode
    ).mean()

    plus_di = 100 * (
        plus_dm_geglaettet / tr_geglaettet
    )

    minus_di = 100 * (
        minus_dm_geglaettet / tr_geglaettet
    )

    dx = 100 * (
        (plus_di - minus_di).abs()
        / (plus_di + minus_di)
    )

    adx = dx.ewm(
        alpha=1 / periode,
        adjust=False,
        min_periods=periode
    ).mean()

    return adx


# ============================================================
# 13. MARKTDATEN VALIDIEREN
# ============================================================

def marktdaten_pruefen(df, coin):

    # Genug historische Kerzen für MA200 + Indikator-Warmlauf

    if len(df) < MIN_ERFORDERLICHE_KERZEN:

        raise ValueError(
            f"Zu wenige historische Kurse: {coin}"
        )

    # Die letzten 50 Stunden müssen lückenlos sein (nach dem
    # Auffüllen von Stunden ohne Handel, siehe Abschnitt 11).

    letzte_kerzen = df.tail(50)

    ohne_handel = letzte_kerzen["ohne_handel"].astype(bool)

    if ohne_handel.sum() > MAX_STUNDEN_OHNE_HANDEL:

        raise ValueError(
            f"Zu wenig Handel: {int(ohne_handel.sum())} der letzten "
            f"50 Stunden ohne Trade ({coin})"
        )

    am_stueck = (
        ohne_handel.groupby((~ohne_handel).cumsum()).sum().max()
    )

    if am_stueck > MAX_LUECKE_AM_STUECK:

        raise ValueError(
            f"Zu lange ohne Handel: {int(am_stueck)} Stunden am "
            f"Stück ({coin})"
        )

    abstaende = (
        letzte_kerzen.index.to_series()
        .diff()
        .dropna()
    )

    if not (
        abstaende == pd.Timedelta(hours=1)
    ).all():

        raise ValueError(
            f"Lücke in den historischen Kursdaten: {coin}"
        )

    # Keine Handelssignale aus veralteten Daten

    aktuelle_stunde = (
        pd.Timestamp.now(tz="UTC").floor("h")
    )

    letzter_zeitpunkt = df.index[-1]

    if letzter_zeitpunkt < (
        aktuelle_stunde - pd.Timedelta(hours=1)
    ):

        raise ValueError(
            f"Historische Kurse veraltet: {coin}"
        )

    return True


# ============================================================
# 14. HANDELSKONTO
# ============================================================

def neues_konto():

    return {

        "startkapital": STARTKAPITAL,

        "guthaben": STARTKAPITAL,

        "positionen": {},

        "transaktionen": [],

        "erstellt_am": zeitstempel(),

        "tagesdatum": None,

        "tagesstartwert": STARTKAPITAL,

        "tageslimit_erreicht": False,

        "letzte_analyse": None,

        "letzter_ausstieg": {}

    }


def konto_pruefen(konto):

    if not isinstance(konto, dict):

        raise ValueError(
            "Ungültige Kontodatei."
        )

    for feld in [
        "startkapital",
        "guthaben",
        "positionen",
        "transaktionen"
    ]:

        if feld not in konto:

            raise ValueError(
                f"Kontofeld fehlt: {feld}"
            )

    if (
        not gueltige_zahl(konto["guthaben"])
        or konto["guthaben"] < -0.000001
    ):

        raise ValueError(
            "Ungültiges Kontoguthaben."
        )

    if not isinstance(
        konto["positionen"], dict
    ):

        raise ValueError(
            "Ungültige Positionsdaten."
        )

    if not isinstance(
        konto["transaktionen"], list
    ):

        raise ValueError(
            "Ungültige Transaktionshistorie."
        )

    for coin, position in konto["positionen"].items():

        if coin not in COINS:

            raise ValueError(
                f"Unbekannte Kryptowährung: {coin}"
            )

        for feld in [
            "menge",
            "einstieg",
            "stop_loss",
            "gesamtkosten",
            "hoechster_schlusskurs",
            "atr_bei_einstieg"
        ]:

            wert = position.get(feld)

            if (
                not gueltige_zahl(wert)
                or wert <= 0
            ):

                raise ValueError(
                    f"Ungültige Position: {coin}, {feld}"
                )

        # Keine Prüfung "stop_loss < einstieg" mehr (V4.0):
        # Der Trailing-Stop steigt in einem guten Trend
        # planmäßig über den Einstieg. V4.0 hat dann beim
        # Speichern einen Fehler geworfen und sich selbst
        # gestoppt. offenes_risiko_berechnen() rechnet einen
        # solchen Stop bereits mit 0 USD Risiko.


def konto_speichern(konto):

    konto_pruefen(konto)

    temporaer = Path(
        str(KONTO_DATEI) + ".tmp"
    )

    with open(
        temporaer,
        "w"
    ) as datei:

        json.dump(
            konto,
            datei,
            indent=4,
            ensure_ascii=False
        )

    os.replace(
        temporaer,
        KONTO_DATEI
    )


def konto_laden():

    if KONTO_DATEI.exists():

        with open(
            KONTO_DATEI,
            "r"
        ) as datei:

            konto = json.load(datei)

        konto_pruefen(konto)

        return konto

    konto = neues_konto()

    konto_speichern(konto)

    return konto


# ============================================================
# 15. BESTEHENDES KONTO ÜBERNEHMEN
# ============================================================

def konto_uebernehmen():

    if (
        KONTO_DATEI.exists()
        and not BACKUP_DATEI.exists()
    ):

        shutil.copy2(
            KONTO_DATEI,
            BACKUP_DATEI
        )

        print(
            "Sicherung des bisherigen Kontos erstellt."
        )

    konto = konto_laden()

    konto.setdefault("tagesdatum", None)

    konto.setdefault(
        "tagesstartwert",
        STARTKAPITAL
    )

    konto.setdefault(
        "tageslimit_erreicht",
        False
    )

    konto.setdefault(
        "letzte_analyse",
        None
    )

    konto.setdefault(
        "letzter_ausstieg",
        {}
    )

    # Verkaufszeitpunkte aus der bisherigen
    # Transaktionshistorie übernehmen.

    for transaktion in konto["transaktionen"]:

        if transaktion.get("aktion") == "VERKAUF":

            coin = transaktion.get("coin")

            verkaufszeit = transaktion.get("zeit")

            if coin in COINS and verkaufszeit:

                bisher = konto["letzter_ausstieg"].get(coin)

                if (
                    bisher is None
                    or verkaufszeit > bisher
                ):

                    konto["letzter_ausstieg"][coin] = (
                        verkaufszeit
                    )

    konto_speichern(konto)

    unterkasten("Handelskonto geladen", "💾")

    zeile("Guthaben", usd(konto["guthaben"]))

    zeile("Offene Positionen", str(len(konto["positionen"])))

    return konto


# ============================================================
# 16. KONTOVERMÖGEN
# ============================================================

def kontowert_berechnen(konto, kurse):

    wert = konto["guthaben"]

    for coin, position in konto["positionen"].items():

        wert += (
            position["menge"]
            * kurse[coin]
        )

    return wert


# ============================================================
# 17. RISIKO OFFENER POSITIONEN
# ============================================================

def offenes_risiko_berechnen(konto):

    gesamtrisiko = 0.0

    for position in konto["positionen"].values():

        stop_erloes = (

            position["menge"]

            * position["stop_loss"]

            * (1 - SLIPPAGE)

            * (1 - GEBUEHR)

        )

        risiko = max(
            0,
            position["gesamtkosten"] - stop_erloes
        )

        gesamtrisiko += risiko

    return gesamtrisiko


# ============================================================
# 18. RISK GATE
# ============================================================

def risk_gate(
    konto,
    coin,
    kurse,
    atr_wert,
    stop_vorgabe=None,
    spread=None
):

    # V6: stop_vorgabe = strategiespezifischer Stop (Funktion
    # von Einstiegskurs -> Stop), spread = relativer Spread.

    # Keine echten Trades

    if not PAPER_TRADING:

        return None, "Nur Paper Trading erlaubt"

    if not AUTOMATISCHE_KAEUFE:

        return None, "Automatische Käufe deaktiviert"

    if konto.get(
        "tageslimit_erreicht",
        False
    ):

        return None, "Tagesverlustlimit erreicht"

    # Keine doppelte Position

    if coin in konto["positionen"]:

        return None, "Position bereits offen"

    if (
        len(konto["positionen"])
        >= MAX_OFFENE_POSITIONEN
    ):

        return None, "Maximale Positionszahl erreicht"

    # Handelspause nach Verkauf

    letzter_ausstieg = konto[
        "letzter_ausstieg"
    ].get(coin)

    if letzter_ausstieg:

        zeitpunkt = datetime.fromisoformat(
            letzter_ausstieg
        )

        if zeitpunkt.tzinfo is None:

            zeitpunkt = zeitpunkt.replace(
                tzinfo=timezone.utc
            )

        if (
            datetime.now(timezone.utc)
            - zeitpunkt
        ).total_seconds() < HANDELSPAUSE:

            return None, "Handelspause nach Verkauf"

    # ATR muss gültig sein

    if (
        not gueltige_zahl(atr_wert)
        or atr_wert <= 0
    ):

        return None, "Ungültiger ATR-Wert"

    # Spread und Handelskosten (Abschnitt 6.4 / 7)

    if spread is None or not gueltige_zahl(spread):

        return None, "Kein Spread bekannt"

    if spread > MAX_SPREAD:

        return None, (
            f"Spread zu hoch ({spread * 100:.2f} % > "
            f"{MAX_SPREAD * 100:.2f} %)"
        )

    # Gesamtes Kontovermögen

    kapital = kontowert_berechnen(
        konto,
        kurse
    )

    # Bereits gebundenes Risiko

    offenes_risiko = offenes_risiko_berechnen(
        konto
    )

    # Verfügbares Gesamtrisiko

    verfuegbares_risiko = (

        kapital * MAX_GESAMTRISIKO

        - offenes_risiko

    )

    # Maximales Risiko je Trade

    risiko_budget = min(

        kapital * MAX_RISIKO_PRO_TRADE,

        verfuegbares_risiko

    )

    if risiko_budget <= 0:

        return None, "Kein Risikobudget"

    # Simulierter Einstiegskurs

    kurs = kurse[coin]

    einstieg = kurs * (1 + SLIPPAGE)

    # Initialer Stop der jeweiligen Strategie

    if stop_vorgabe is None:

        stop_loss = einstieg - (
            ATR_STOP_MULTIPLIKATOR * atr_wert
        )

    else:

        stop_loss = stop_vorgabe(einstieg)

    if (
        not gueltige_zahl(stop_loss)
        or stop_loss <= 0
    ):

        return None, "Ungültiger Stop"

    if stop_loss >= einstieg:

        return None, "Stop liegt nicht unter dem Einstieg"

    # Risiko je Einheit inklusive Kosten

    stop_erloes = (

        stop_loss

        * (1 - SLIPPAGE)

        * (1 - GEBUEHR)

    )

    risiko_je_einheit = (

        einstieg * (1 + GEBUEHR)

        - stop_erloes

    )

    if risiko_je_einheit <= 0:

        return None, "Ungültige Risikoberechnung"

    # Positionsgröße nach Risiko

    menge_risiko = (

        risiko_budget

        / risiko_je_einheit

    )

    # Positionsgröße nach maximalem Positionswert

    menge_positionslimit = (

        kapital * MAX_POSITIONSANTEIL

    ) / (

        einstieg * (1 + GEBUEHR)

    )

    # Verfügbares Guthaben berücksichtigen

    menge_guthaben = (

        konto["guthaben"]

    ) / (

        einstieg * (1 + GEBUEHR)

    )

    menge = min(

        menge_risiko,

        menge_positionslimit,

        menge_guthaben

    )

    if (
        not gueltige_zahl(menge)
        or menge <= 0
    ):

        return None, "Ungültige Positionsgröße"

    geplantes_risiko = (
        menge * risiko_je_einheit
    )

    # Abschließende Risikoprüfung

    if geplantes_risiko > (
        kapital * MAX_RISIKO_PRO_TRADE + 1e-8
    ):

        return None, "Einzelrisiko überschritten"

    if (
        offenes_risiko + geplantes_risiko
        > kapital * MAX_GESAMTRISIKO + 1e-8
    ):

        return None, "Gesamtrisiko überschritten"

    return {

        "menge": menge,

        "einstieg": einstieg,

        "stop_loss": stop_loss,

        "atr_bei_einstieg": atr_wert,

        "hoechster_schlusskurs": kurs,

        "geplantes_risiko": geplantes_risiko

    }, "Freigegeben"


# ============================================================
# 19. VIRTUELLER KAUF
# ============================================================

def virtueller_kauf(konto, coin, pruefung):

    menge = pruefung["menge"]

    einstieg = pruefung["einstieg"]

    positionswert = (
        menge * einstieg
    )

    gebuehr = (
        positionswert * GEBUEHR
    )

    gesamtkosten = (
        positionswert + gebuehr
    )

    if coin in konto["positionen"]:

        print(
            faerben(
                "⚠️  Kauf abgelehnt: Position bereits offen.",
                FARBE_GELB
            )
        )

        return False

    if gesamtkosten > konto["guthaben"] + 1e-8:

        print(
            faerben(
                "⚠️  Kauf abgelehnt: Guthaben zu gering.",
                FARBE_GELB
            )
        )

        return False

    # Guthaben reduzieren

    konto["guthaben"] -= gesamtkosten

    # Position anlegen

    konto["positionen"][coin] = {

        "menge": menge,

        "einstieg": einstieg,

        "stop_loss": pruefung["stop_loss"],

        "atr_bei_einstieg": pruefung["atr_bei_einstieg"],

        "hoechster_schlusskurs": pruefung["hoechster_schlusskurs"],

        "gesamtkosten": gesamtkosten,

        "kaufzeit": zeitstempel(),

        "datenquelle": "Alpaca",

        # V6.2: Risiko bis zum Anfangs-Stop inkl. Kosten in USD
        # (= 1 R). Der Stop wandert später mit, dieser Wert nicht.
        "anfangsrisiko": pruefung["geplantes_risiko"],

        **pruefung.get("zusatz", {})

    }

    # Transaktion dokumentieren

    konto["transaktionen"].append({

        "zeit": zeitstempel(),

        "coin": coin,

        "aktion": "KAUF",

        "menge": menge,

        "kurs": einstieg,

        "gebuehr": gebuehr,

        "gesamtkosten": gesamtkosten,

        "atr_bei_einstieg": pruefung["atr_bei_einstieg"],

        "anfangsrisiko": pruefung["geplantes_risiko"],

        "strategie": pruefung.get("zusatz", {}).get("strategie"),

        "datenquelle": "Alpaca"

    })

    konto_speichern(konto)

    strategie = pruefung.get("zusatz", {}).get("strategie")

    ereignis(
        "KAUF",
        coin,
        f"{STRATEGIE_NAMEN.get(strategie, '')} · {usd(einstieg)} · "
        f"Stop {usd(pruefung['stop_loss'])}"
    )

    abstand_stop = einstieg - pruefung["stop_loss"]

    abstand_stop_prozent = (
        abstand_stop / einstieg * 100
    )

    box_start(f"KAUF: {COINS[coin]['name']}", "✅")

    box_zeile(f"Kaufkurs        {usd(einstieg)}")

    box_zeile(f"Positionswert   {usd(positionswert)}")

    box_zeile(
        f"Risiko          "
        f"{usd(pruefung['geplantes_risiko'])}"
    )

    box_zeile(
        f"Stop-Loss       {usd(pruefung['stop_loss'])}  "
        f"(-{usd(abstand_stop)} / -{abstand_stop_prozent:.2f} %)"
    )

    box_zeile("Take-Profit     kein festes Ziel (Trailing-Stop)")

    box_zeile(
        f"ATR bei Einstieg {usd(pruefung['atr_bei_einstieg'])}"
    )

    box_ende()

    return True


# ============================================================
# 20. VIRTUELLER VERKAUF
# ============================================================

GRUND_TEXTE = {
    "STOP": "Stop ausgelöst",
    "TAGESLIMIT": "Tagesverlustlimit erreicht",
    "TRENDBRUCH": "A: Stundenschluss unter EMA 50",
    "FEHLAUSBRUCH": "B: Stundenschluss zurück in der Spanne",
    "ZEITSTOP": "B: Zeitstop ohne Fortschritt"
}

def virtueller_verkauf(
    konto,
    coin,
    referenzkurs,
    grund
):

    position = konto["positionen"][coin]

    menge = position["menge"]

    verkaufskurs = (
        referenzkurs * (1 - SLIPPAGE)
    )

    verkaufswert = (
        menge * verkaufskurs
    )

    gebuehr = (
        verkaufswert * GEBUEHR
    )

    netto_erloes = (
        verkaufswert - gebuehr
    )

    gewinn = (
        netto_erloes
        - position["gesamtkosten"]
    )

    # V6.2: Ergebnis in Vielfachen des Anfangsrisikos (R).
    # Positionen aus älteren Konten ohne Anfangsrisiko: None.

    anfangsrisiko = position.get("anfangsrisiko")

    r_vielfaches = (
        gewinn / anfangsrisiko
        if anfangsrisiko else None
    )

    # Guthaben aktualisieren

    konto["guthaben"] += netto_erloes

    # Position entfernen

    del konto["positionen"][coin]

    # Zeitpunkt des Verkaufs speichern

    konto["letzter_ausstieg"][coin] = (
        zeitstempel()
    )

    # Transaktion dokumentieren

    konto["transaktionen"].append({

        "zeit": zeitstempel(),

        "coin": coin,

        "aktion": "VERKAUF",

        "grund": grund,

        "strategie": position.get("strategie"),

        "menge": menge,

        "kurs": verkaufskurs,

        "gebuehr": gebuehr,

        "netto_erloes": netto_erloes,

        "gewinn": gewinn,

        "anfangsrisiko": anfangsrisiko,

        "r_vielfaches": r_vielfaches,

        "datenquelle": "Alpaca"

    })

    konto_speichern(konto)

    grund_text = GRUND_TEXTE.get(grund, grund)

    ereignis(
        "VERKAUF",
        coin,
        f"{grund_text} · {usd(verkaufskurs)} · "
        f"Ergebnis {usd(gewinn)}"
    )

    box_start(f"VERKAUF: {COINS[coin]['name']}", "🛑")

    box_zeile(f"Grund           {grund_text}")

    box_zeile(f"Verkaufskurs    {usd(verkaufskurs)}")

    box_zeile(
        "Gewinn/Verlust  "
        + wert_faerben(gewinn, usd(gewinn))
        + f"  {pfeil(gewinn)}"
    )

    box_ende()


# ============================================================
# 20b. AUSWERTUNG JE STRATEGIE (V6.2)
# ============================================================
#
# Nur Anzeige: fasst die abgeschlossenen Trades je Strategie
# zusammen. Der Bot ändert daraufhin nichts; über Parameter
# entscheidet der Mensch nach genügend Trades.

def strategie_auswertung(transaktionen):

    gruppen = {}

    for t in transaktionen:

        if t.get("aktion") != "VERKAUF":
            continue

        schluessel = t.get("strategie") or "?"

        g = gruppen.setdefault(schluessel, {
            "trades": 0,
            "gewinner": 0,
            "ergebnis": 0.0,
            "r_werte": [],
            "gruende": {}
        })

        gewinn = t.get("gewinn") or 0.0

        g["trades"] += 1
        g["gewinner"] += gewinn > 0
        g["ergebnis"] += gewinn

        if t.get("r_vielfaches") is not None:
            g["r_werte"].append(t["r_vielfaches"])

        grund = t.get("grund") or "?"
        g["gruende"][grund] = g["gruende"].get(grund, 0) + 1

    ergebnis = {}

    for schluessel in sorted(gruppen):

        g = gruppen[schluessel]
        r_werte = g.pop("r_werte")

        ergebnis[schluessel] = {
            **g,
            "trefferquote": g["gewinner"] / g["trades"],
            "durchschnitt": g["ergebnis"] / g["trades"],
            "r_anzahl": len(r_werte),
            "r_durchschnitt": (
                sum(r_werte) / len(r_werte) if r_werte else None
            )
        }

    return ergebnis


# ============================================================
# 21. TRAILING-STOP AKTUALISIEREN
# ============================================================

def trailing_stop_berechnen(
    position,
    letzter_schluss,
    aktueller_atr
):

    # Reine Berechnung ohne Seiteneffekt (kein Speichern) -
    # so lässt sich die Trailing-Stop-Logik im Funktionstest
    # prüfen, ohne die echte Kontodatei anzufassen.

    hoechster = position["hoechster_schlusskurs"]

    stop = position["stop_loss"]

    if (
        not gueltige_zahl(letzter_schluss)
        or not gueltige_zahl(aktueller_atr)
        or aktueller_atr <= 0
    ):

        return hoechster, stop, False

    # Trailing-Anker nur nach oben anpassen

    if letzter_schluss > hoechster:

        hoechster = letzter_schluss

    neuer_stop = (

        hoechster

        - (ATR_TRAILING_MULTIPLIKATOR * aktueller_atr)

    )

    # Stop darf nur steigen, nie sinken

    if neuer_stop > stop:

        return hoechster, neuer_stop, True

    return hoechster, stop, False


def trailing_stop_aktualisieren(
    konto,
    coin,
    letzter_schluss,
    aktueller_atr
):

    if coin not in konto["positionen"]:

        return

    position = konto["positionen"][coin]

    alter_stop = position["stop_loss"]

    alter_hoechster = position["hoechster_schlusskurs"]

    hoechster, neuer_stop, geaendert = trailing_stop_berechnen(
        position,
        letzter_schluss,
        aktueller_atr
    )

    position["hoechster_schlusskurs"] = hoechster

    if geaendert:

        position["stop_loss"] = neuer_stop

    # Auch einen neuen Höchstkurs ohne Stop-Änderung speichern
    # (V4.0 hat ihn dann verworfen, weil das Konto in jedem
    # Durchlauf neu aus der Datei geladen wird).

    if geaendert or hoechster != alter_hoechster:

        konto_speichern(konto)

    if geaendert:

        ereignis(
            "STOP_NACHGEZOGEN",
            coin,
            f"{usd(alter_stop)} → {usd(neuer_stop)}"
        )

        print(
            faerben(
                f"  ↳ 🔧 Trailing-Stop {COINS[coin]['name']} "
                f"nachgezogen: {usd(alter_stop)} → "
                f"{usd(neuer_stop)}",
                FARBE_GELB
            )
        )


# ============================================================
# 22. POSITIONEN ÜBERWACHEN
# ============================================================

def positionen_ueberwachen(konto, kurse):

    if not konto["positionen"]:

        return

    ausgestoppte_coins = []

    box_start("Positions-Check", "🔍")

    box_zeile(
        f"{'Coin':<10}{'Kurs':<12}{'Stop-Loss':<13}"
        f"{'Abstand':<10}Status"
    )

    for coin in list(
        konto["positionen"].keys()
    ):

        position = konto["positionen"][coin]

        kurs = kurse[coin]

        stop = position["stop_loss"]

        abstand_stop_prozent = (
            (kurs - stop) / kurs * 100
            if kurs else 0.0
        )

        ausgestoppt = kurs <= stop

        if ausgestoppt:

            ausgestoppte_coins.append(coin)

            status = faerben("🛑 STOP", FARBE_ROT)

        else:

            status = faerben("✅ offen", FARBE_GRUEN)

        box_zeile(
            f"{COINS[coin]['name']:<10}"
            f"{usd(kurs):<12}"
            f"{usd(stop):<13}"
            f"{abstand_stop_prozent:>5.2f} %   "
            f"{status}"
        )

    box_ende()

    # Verkäufe nach der Tabellenanzeige ausführen, damit jede
    # ausgelöste Verkaufsbox darunter erscheint statt die
    # Übersicht zu unterbrechen.

    for coin in ausgestoppte_coins:

        virtueller_verkauf(
            konto,
            coin,
            kurse[coin],
            "STOP"
        )


# ============================================================
# 23. TAGESVERLUSTLIMIT
# ============================================================

def tageslimit_pruefen(konto, kurse):

    heute = datetime.now(
        timezone.utc
    ).date().isoformat()

    kapital = kontowert_berechnen(
        konto,
        kurse
    )

    # Tagesstartwert speichern

    if konto.get("tagesdatum") != heute:

        konto["tagesdatum"] = heute

        konto["tagesstartwert"] = kapital

        konto["tageslimit_erreicht"] = False

        konto_speichern(konto)

    verlustgrenze = (

        konto["tagesstartwert"]

        * (1 - MAX_TAGESVERLUST)

    )

    if kapital <= verlustgrenze:

        if not konto["tageslimit_erreicht"]:

            konto["tageslimit_erreicht"] = True

            konto_speichern(konto)

            kasten("TAGESVERLUSTLIMIT ERREICHT", "⚠️")

            ereignis(
                "TAGESLIMIT",
                None,
                "Alle Positionen geschlossen, kein Handel bis morgen (UTC)"
            )

            print(
                faerben(
                    "Alle offenen Positionen werden "
                    "geschlossen. Kein Handel bis morgen.",
                    FARBE_ROT
                )
            )

            # Alle Positionen virtuell schließen

            for coin in list(
                konto["positionen"].keys()
            ):

                virtueller_verkauf(
                    konto,
                    coin,
                    kurse[coin],
                    "TAGESLIMIT"
                )

        return False

    return not konto["tageslimit_erreicht"]


# ============================================================
# 24. MARKTLAGE UND STRATEGIEN A / B
# ============================================================

# Ablauf:
# - jede volle Stunde und bei jedem Start: Stundenkerzen holen,
#   Marktlage je Coin bestimmen, Pullback (A) und Squeeze (B)
#   erkennen, Ausstiege offener Positionen prüfen
# - alle 15 Minuten: 15-Minuten-Kerzen holen und für Coins mit
#   bereitem Pullback bzw. aktivem Squeeze das Einstiegssignal
#   prüfen; gültige Signale gehen an das Risk Gate
# Der Zustand von A/B je Coin wird gespeichert, damit ein
# Neustart einen laufenden Squeeze nicht vergisst.

REGIME_TEXTE = {
    "AUFWAERTSTREND": "Aufwärtstrend",
    "ABWAERTSTREND": "Abwärtstrend",
    "NEUTRAL": "neutral"
}


def indikatoren_1h(df):

    df = df.copy()

    for periode in (EMA_KURZ, EMA_MITTEL, EMA_LANG):

        df[f"EMA{periode}"] = df["close"].ewm(
            span=periode,
            adjust=False,
            min_periods=periode
        ).mean()

    df["ATR"] = atr_berechnen(df, ATR_PERIODE)

    df["ADX"] = adx_berechnen(df, ADX_PERIODE)

    mitte = df["close"].rolling(BB_PERIODE).mean()

    abweichung = df["close"].rolling(BB_PERIODE).std(ddof=0)

    # Relative Bollinger-Bandbreite (oberes - unteres) / mittleres

    df["BBW"] = (2 * BB_STDABW * abweichung) / mitte

    return df


def marktregime(kerze):

    # VORLÄUFIG (Abschnitt 4.1 / 4.3 fehlen noch)

    ema_mittel = kerze[f"EMA{EMA_MITTEL}"]

    ema_lang = kerze[f"EMA{EMA_LANG}"]

    if ema_mittel < ema_lang and kerze["close"] < ema_lang:

        return "ABWAERTSTREND"

    if ema_mittel > ema_lang and kerze["ADX"] >= ADX_MINDESTWERT:

        return "AUFWAERTSTREND"

    return "NEUTRAL"


def pullback_pruefen(df):

    # Strategie A, Prüfungen 1 und 2 auf abgeschlossenen
    # Stundenkerzen. Gibt (pullback, grund) zurück.

    letzte = df.iloc[-1]

    ema_kurz = f"EMA{EMA_KURZ}"

    ema_mittel = f"EMA{EMA_MITTEL}"

    if marktregime(letzte) != "AUFWAERTSTREND":

        return None, "kein Aufwärtstrend"

    if not letzte["close"] > letzte[ema_mittel]:

        return None, f"Schluss unter EMA {EMA_MITTEL}"

    fenster = df.iloc[-PULLBACK_KERZEN:]

    davor = df.iloc[-PULLBACK_KERZEN - 1:-1]

    if (fenster["close"] < fenster[ema_mittel]).any():

        return None, f"Pullback schloss unter EMA {EMA_MITTEL}"

    # "von oben erreicht oder unterschritten": Tief der Kerze
    # <= EMA20, die Kerze davor schloss über dem EMA20

    beruehrt = (
        (fenster["low"].to_numpy() <= fenster[ema_kurz].to_numpy())
        & (davor["close"].to_numpy() > davor[ema_kurz].to_numpy())
    )

    if not beruehrt.any():

        return None, f"kein Rücksetzer an EMA {EMA_KURZ}"

    if not letzte["close"] > letzte[ema_kurz]:

        return None, f"noch nicht zurück über EMA {EMA_KURZ}"

    letzte_beruehrung = fenster.index[
        len(beruehrt) - 1 - beruehrt[::-1].argmax()
    ]

    return {
        "id": letzte_beruehrung.isoformat(),
        "tief": float(fenster["low"].min())
    }, "Pullback erkannt"


def squeeze_pruefen(df):

    # Strategie B, 6.2: aktuelle Bandbreite unter dem 20-%-
    # Quantil der 120 VORHERIGEN Bandbreiten (der aktuelle Wert
    # geht nicht in seine eigene Schwelle ein).

    bandbreiten = df["BBW"]

    aktuell = float(bandbreiten.iloc[-1])

    vergleich = bandbreiten.iloc[-1 - SQUEEZE_REFERENZ:-1]

    if (
        len(vergleich) < SQUEEZE_REFERENZ
        or vergleich.isna().any()
        or not gueltige_zahl(aktuell)
    ):

        return False, {}, "zu wenig Daten"

    schwelle = float(vergleich.quantile(SQUEEZE_QUANTIL))

    info = {"bbw": aktuell, "schwelle": schwelle}

    # Flache Kerzen ohne Handel drücken die Bandbreite künstlich;
    # V6.3 duldet einzelne davon (Alpaca handelt manche Coins dünn)

    if (
        df["ohne_handel"].iloc[-BB_PERIODE:].astype(bool).sum()
        > B_MAX_STUNDEN_OHNE_HANDEL
    ):

        return False, info, "zu viele Stunden ohne Handel in der Bandbreite"

    if aktuell > schwelle:

        return False, info, "keine Kompression"

    return True, info, "Kompression"


def zustand_a_aktualisieren(a, df, kerzenende):

    a = dict(a or {})

    benutzt = list(a.get("benutzt", []))[-20:]

    pullback, grund = pullback_pruefen(df)

    if pullback is None:

        return {"status": "INAKTIV", "grund": grund, "benutzt": benutzt}

    if pullback["id"] in benutzt:

        return {"status": "BENUTZT", "grund": "Pullback bereits gehandelt",
                "benutzt": benutzt, **pullback}

    if a.get("id") == pullback["id"] and a.get("seit"):

        seit = a["seit"]

    else:

        seit = kerzenende.isoformat()

    bis = parse_utc(seit) + timedelta(hours=PULLBACK_GUELTIG_STUNDEN)

    status = "BEREIT" if kerzenende < bis else "ABGELAUFEN"

    return {"status": status, "grund": grund, "benutzt": benutzt,
            "seit": seit, "bis": bis.isoformat(), **pullback}


def zustand_b_aktualisieren(b, df, kerzenende):

    b = dict(b or {"status": "INAKTIV"})

    squeeze, info, grund = squeeze_pruefen(df)

    abwaerts = marktregime(df.iloc[-1]) == "ABWAERTSTREND"

    if squeeze and abwaerts:

        squeeze, grund = False, "Abwärtstrend"

    if b.get("status") == "AKTIV" and kerzenende >= parse_utc(b["bis"]):

        b["status"] = "ABGELAUFEN"

    # Nach Ausbruch oder Ablauf erst eine neue Phase, wenn die
    # Kompression zwischendurch geendet hat (6.5)

    if b.get("status") in ("BENUTZT", "ABGELAUFEN") and not squeeze:

        b = {"status": "INAKTIV"}

    if b.get("status") == "INAKTIV" and squeeze:

        spanne = df.iloc[-SPANNE_KERZEN:]

        b = {
            "status": "AKTIV",
            "oben": float(spanne["high"].max()),
            "unten": float(spanne["low"].min()),
            "seit": kerzenende.isoformat(),
            "bis": (kerzenende + timedelta(hours=SQUEEZE_GUELTIG_STUNDEN)).isoformat()
        }

    b.update(info)

    b["grund"] = grund

    return b


# ---------- Blocker-Statistik (V6.3) ----------

# Zählt je Prüfstufe, woran ein möglicher Einstieg gescheitert ist
# (bzw. dass er durchkam). Nur zur Anzeige und für die nächste
# Überprüfung der Parameter; der Handel hängt nicht davon ab.

BLOCKER_STUFEN = ("daten", "a", "b", "signal_a", "signal_b", "risk_gate")


def blocker_grund(text):

    # Zahlen und Coin-Namen weglassen, damit gleiche Gründe
    # zusammen gezählt werden: "Spread zu hoch (0.20 % > ...)"
    # -> "Spread zu hoch", "Zu wenig Handel: 7 der ..." -> "Zu wenig Handel"

    text = re.sub(r"\s*\(.*?\)", "", str(text or ""))

    return text.split(":")[0].strip() or "?"


def blocker_zaehlen(statistik, stufe, grund):

    zaehler = statistik.setdefault(stufe, {})

    grund = blocker_grund(grund)

    zaehler[grund] = zaehler.get(grund, 0) + 1


def blocker_statistik_laden():

    if STATISTIK_DATEI is None or not STATISTIK_DATEI.exists():

        return {"seit": zeitstempel()}

    try:

        daten = json.loads(STATISTIK_DATEI.read_text(encoding="utf-8"))

    except (OSError, ValueError):

        return {"seit": zeitstempel()}

    return daten if isinstance(daten, dict) else {"seit": zeitstempel()}


def blocker_statistik_speichern(statistik):

    if STATISTIK_DATEI is None:

        return

    temporaer = Path(str(STATISTIK_DATEI) + ".tmp")

    temporaer.write_text(
        json.dumps(statistik, ensure_ascii=False, indent=2),
        encoding="utf-8"
    )

    os.replace(temporaer, STATISTIK_DATEI)


A_STATUS_TEXTE = {
    "BEREIT": "Pullback bereit",
    "BENUTZT": "Pullback bereits gehandelt",
    "ABGELAUFEN": "Pullback-Fenster abgelaufen"
}

B_STATUS_TEXTE = {
    "AKTIV": "Squeeze aktiv",
    "BENUTZT": "Squeeze bereits gehandelt",
    "ABGELAUFEN": "Squeeze abgelaufen"
}


def parse_utc(wert):

    zeitpunkt = datetime.fromisoformat(wert) if isinstance(wert, str) else wert

    if zeitpunkt.tzinfo is None:

        zeitpunkt = zeitpunkt.replace(tzinfo=timezone.utc)

    return zeitpunkt.astimezone(timezone.utc)


def signal_15m_pruefen(df15, seit):

    # Gemeinsame Prüfung der zuletzt abgeschlossenen
    # 15-Minuten-Kerze. Gibt (werte, grund) zurück; werte=None
    # heißt: kein auswertbares Signal.

    if len(df15) < VOLUMEN_VERGLEICH_KERZEN + 1:

        return None, "zu wenige 15-Min-Kerzen"

    kerze = df15.iloc[-1]

    ende = df15.index[-1] + pd.Timedelta(minutes=15)

    if ende <= pd.Timestamp(parse_utc(seit)):

        return None, "noch keine neue 15-Min-Kerze"

    if bool(kerze["ohne_handel"]):

        return None, "letzte 15-Min-Kerze ohne Handel"

    vergleich = df15.iloc[-VOLUMEN_VERGLEICH_KERZEN - 1:-1]

    if vergleich["ohne_handel"].astype(bool).sum() > MAX_15M_OHNE_HANDEL:

        return None, "zu wenig Handel in den 15-Min-Kerzen"

    return {
        "kerze_utc": df15.index[-1].isoformat(),
        "schluss": float(kerze["close"]),
        "volumen": float(kerze["volume"]),
        "volumen_schnitt": float(vergleich["volume"].mean()),
        "hoch_vorher": float(
            df15["high"].iloc[-A_AUSBRUCH_KERZEN - 1:-1].max()
        )
    }, ""


def signal_a_pruefen(df15, a):

    werte, grund = signal_15m_pruefen(df15, a["seit"])

    if werte is None:

        return None, grund

    if pd.Timestamp(parse_utc(werte["kerze_utc"])) >= pd.Timestamp(parse_utc(a["bis"])):

        return None, "Pullback-Fenster abgelaufen"

    if not werte["schluss"] > werte["hoch_vorher"]:

        return None, (
            f"Schluss nicht über Hoch der letzten "
            f"{A_AUSBRUCH_KERZEN} Kerzen"
        )

    if werte["volumen"] < A_VOLUMEN_FAKTOR * werte["volumen_schnitt"]:

        return None, "Volumen zu gering"

    return werte, "Signal A"


def signal_b_pruefen(df15, b, atr):

    werte, grund = signal_15m_pruefen(df15, b["seit"])

    if werte is None:

        return None, grund

    if not werte["schluss"] > b["oben"]:

        return None, "kein Schluss über der Spanne"

    if werte["volumen"] < B_VOLUMEN_FAKTOR * werte["volumen_schnitt"]:

        return None, "Ausbruchsvolumen zu gering"

    if werte["schluss"] > b["oben"] + B_UEBERDEHNUNG_ATR * atr:

        return None, "Ausbruch schon zu weit gelaufen"

    return werte, "Signal B"


# ---------- gespeicherter Zustand A/B ----------

def strategie_zustand_laden():

    if STRATEGIE_DATEI is None or not STRATEGIE_DATEI.exists():

        return {}

    try:

        daten = json.loads(STRATEGIE_DATEI.read_text(encoding="utf-8"))

    except (OSError, ValueError):

        return {}

    return daten if isinstance(daten, dict) else {}


def strategie_zustand_speichern(zustand):

    temporaer = Path(str(STRATEGIE_DATEI) + ".tmp")

    temporaer.write_text(
        json.dumps(zustand, ensure_ascii=False, indent=2),
        encoding="utf-8"
    )

    os.replace(temporaer, STRATEGIE_DATEI)


# ---------- stündliche Marktanalyse ----------

def stundenanalyse(konto, kurse, lauf):

    kasten("MARKTLAGE (STÜNDLICH)", "📊")

    zustand = lauf["strategien"]

    marktlage = lauf["marktlage"]

    indikatoren = {}

    for coin in HANDELSPAARE:

        eintrag = marktlage.setdefault(coin, {"coin": coin})

        eintrag.update(zeit=zeitstempel(), status="OK", fehler=None)

        coin_zustand = zustand.setdefault(coin, {})

        try:

            df = historische_kurse(coin, "1h")

            marktdaten_pruefen(df, coin)

        except (
            requests.exceptions.RequestException,
            APIError,
            RuntimeError,
            ValueError
        ) as e:

            eintrag.update(status="DATENFEHLER", fehler=str(e))

            ereignis("DATENFEHLER", coin, str(e))

            blocker_zaehlen(lauf["statistik"], "daten", str(e))

            # Ohne verlässliche Daten kein bereiter Einstieg

            coin_zustand["a"] = {"status": "INAKTIV", "grund": "Datenfehler",
                                 "benutzt": (coin_zustand.get("a") or {}).get("benutzt", [])}

            if (coin_zustand.get("b") or {}).get("status") == "AKTIV":

                coin_zustand["b"]["status"] = "ABGELAUFEN"

            print(faerben(f"  ⚠️  {COINS[coin]['name']}: {e}", FARBE_GELB))

            continue

        df = indikatoren_1h(df)

        letzte = df.iloc[-1]

        kerzenende = (df.index[-1] + pd.Timedelta(hours=1)).to_pydatetime()

        indikatoren[coin] = letzte

        regime = marktregime(letzte)

        coin_zustand["a"] = zustand_a_aktualisieren(coin_zustand.get("a"), df, kerzenende)

        coin_zustand["b"] = zustand_b_aktualisieren(coin_zustand.get("b"), df, kerzenende)

        blocker_zaehlen(lauf["statistik"], "daten", "Daten in Ordnung")

        a_neu, b_neu = coin_zustand["a"], coin_zustand["b"]

        blocker_zaehlen(lauf["statistik"], "a",
                        A_STATUS_TEXTE.get(a_neu["status"], a_neu.get("grund")))

        blocker_zaehlen(lauf["statistik"], "b",
                        B_STATUS_TEXTE.get(b_neu["status"], b_neu.get("grund")))

        eintrag.update(
            kerze_utc=df.index[-1].isoformat(),
            regime=regime,
            letzter_schluss=float(letzte["close"]),
            ema_kurz=float(letzte[f"EMA{EMA_KURZ}"]),
            ema_mittel=float(letzte[f"EMA{EMA_MITTEL}"]),
            ema_lang=float(letzte[f"EMA{EMA_LANG}"]),
            adx=float(letzte["ADX"]),
            atr=float(letzte["ATR"]),
            ohne_handel_50h=int(df.tail(50)["ohne_handel"].astype(bool).sum()),
            a=coin_zustand["a"],
            b=coin_zustand["b"]
        )

        print(
            f"  {COINS[coin]['name']:<10} {REGIME_TEXTE[regime]:<14} "
            f"A: {coin_zustand['a']['status']:<10} "
            f"B: {coin_zustand['b']['status']}"
        )

        # Ausstiege und Trailing-Stop offener Positionen

        if coin in konto["positionen"]:

            ausstieg_pruefen(konto, coin, kurse, df)

    strategie_zustand_speichern(zustand)

    blocker_statistik_speichern(lauf["statistik"])

    lauf["atr"] = {coin: float(z["ATR"]) for coin, z in indikatoren.items()}

    lauf["regime"] = {coin: marktlage[coin].get("regime") for coin in indikatoren}

    analyse_speichern(list(marktlage.values()))


def ausstieg_pruefen(konto, coin, kurse, df):

    position = konto["positionen"][coin]

    letzte = df.iloc[-1]

    atr = float(letzte["ATR"])

    trailing_stop_aktualisieren(konto, coin, float(letzte["close"]), atr)

    if coin not in kurse:

        return

    kerzenende = df.index[-1] + pd.Timedelta(hours=1)

    nach_kauf = kerzenende > pd.Timestamp(parse_utc(position["kaufzeit"]))

    strategie = position.get("strategie")

    grund = None

    if (
        strategie == "A"
        and nach_kauf
        and letzte["close"] < letzte[f"EMA{EMA_MITTEL}"]
    ):

        grund = "TRENDBRUCH"

    elif strategie == "B":

        alter = (
            datetime.now(timezone.utc) - parse_utc(position["kaufzeit"])
        ).total_seconds() / 3600

        if nach_kauf and letzte["close"] < position.get("spanne_oben", 0):

            grund = "FEHLAUSBRUCH"

        elif (
            alter >= B_ZEITSTOP_STUNDEN
            and position["hoechster_schlusskurs"]
            < position["einstieg"] + position["atr_bei_einstieg"]
        ):

            grund = "ZEITSTOP"

    if grund:

        virtueller_verkauf(konto, coin, kurse[coin], grund)


# ---------- Einstiegssignale alle 15 Minuten ----------

def signalpruefung(konto, kurse, lauf):

    zustand = lauf["strategien"]

    kandidaten = []

    for coin in HANDELSPAARE:

        coin_zustand = zustand.get(coin, {})

        a = coin_zustand.get("a") or {}

        b = coin_zustand.get("b") or {}

        eintrag = lauf["marktlage"].setdefault(coin, {"coin": coin})

        if a.get("status") != "BEREIT" and b.get("status") != "AKTIV":

            eintrag["signal"] = None

            continue

        try:

            df15 = historische_kurse(coin, "15m")

        except (
            requests.exceptions.RequestException,
            APIError,
            RuntimeError,
            ValueError
        ) as e:

            eintrag["signal"] = {"zeit": zeitstempel(), "text": f"Datenfehler 15 Min: {e}"}

            continue

        atr = lauf.get("atr", {}).get(coin)

        texte = []

        if a.get("status") == "BEREIT" and atr:

            werte, grund = signal_a_pruefen(df15, a)

            texte.append(f"A: {grund}")

            blocker_zaehlen(lauf["statistik"], "signal_a", grund)

            if werte is not None and lauf["regime"].get(coin) == "AUFWAERTSTREND":

                stop_basis = a["tief"] - A_STOP_ATR * atr

                kandidaten.append(("A", coin, werte, lambda einstieg, s=stop_basis: s,
                                   {"strategie": "A", "pullback_id": a["id"],
                                    "pullback_tief": a["tief"]}))

        if b.get("status") == "AKTIV" and atr:

            werte, grund = signal_b_pruefen(df15, b, atr)

            texte.append(f"B: {grund}")

            blocker_zaehlen(lauf["statistik"], "signal_b", grund)

            if werte is not None and lauf["regime"].get(coin) != "ABWAERTSTREND":

                mitte = (b["oben"] + b["unten"]) / 2

                kandidaten.append(("B", coin, werte,
                                   lambda einstieg, m=mitte, a_=atr: max(m, einstieg - B_STOP_ATR * a_),
                                   {"strategie": "B", "spanne_oben": b["oben"],
                                    "spanne_unten": b["unten"]}))

        eintrag["signal"] = {"zeit": zeitstempel(), "text": " · ".join(texte)}

    blocker_statistik_speichern(lauf["statistik"])

    if not kandidaten:

        return

    # Rangfolge: geringstes Risiko je Einheit zuerst

    def risiko_anteil(kandidat):

        strategie, coin, werte, stop_vorgabe, _ = kandidat

        if coin not in kurse:

            return float("inf")

        einstieg = kurse[coin] * (1 + SLIPPAGE)

        return (einstieg - stop_vorgabe(einstieg)) / einstieg

    kandidaten.sort(key=risiko_anteil)

    kasten("EINSTIEGSSIGNALE (15 MIN)", "🎯")

    for strategie, coin, werte, stop_vorgabe, zusatz in kandidaten:

        name = f"{COINS[coin]['name']} · {STRATEGIE_NAMEN[strategie]}"

        if coin not in kurse:

            ereignis("KEIN_KAUF", coin, f"{STRATEGIE_NAMEN[strategie]}: kein aktueller Kurs")

            continue

        pruefung, grund = risk_gate(
            konto, coin, kurse, lauf["atr"][coin],
            stop_vorgabe=stop_vorgabe,
            spread=SPREADS.get(coin)
        )

        eintrag = lauf["marktlage"][coin]

        blocker_zaehlen(lauf["statistik"], "risk_gate", grund)

        if pruefung is None:

            print(faerben(f"  ⚠️  Kein Kauf ({name}): {grund}", FARBE_GELB))

            eintrag["signal"]["text"] += f" → kein Kauf: {grund}"

            if grund not in ROUTINE_ABLEHNUNGEN:

                ereignis("KEIN_KAUF", coin, f"{STRATEGIE_NAMEN[strategie]}: {grund}")

            continue

        pruefung["zusatz"] = zusatz

        if virtueller_kauf(konto, coin, pruefung):

            eintrag["signal"]["text"] += " → virtuell gekauft"

            # Dieselbe Gelegenheit nicht erneut handeln (5.x / 6.5)

            coin_zustand = lauf["strategien"][coin]

            if strategie == "A":

                coin_zustand["a"]["benutzt"] = (
                    coin_zustand["a"].get("benutzt", []) + [zusatz["pullback_id"]]
                )[-20:]

                coin_zustand["a"]["status"] = "BENUTZT"

            else:

                coin_zustand["b"]["status"] = "BENUTZT"

            strategie_zustand_speichern(lauf["strategien"])

    blocker_statistik_speichern(lauf["statistik"])


ROUTINE_ABLEHNUNGEN = (
    "Position bereits offen",
    "Maximale Positionszahl erreicht",
    "Handelspause nach Verkauf",
    "Tagesverlustlimit erreicht"
)


def analyse_speichern(ergebnisse):

    # Nur Anzeige - ein Schreibfehler darf den Handel nicht
    # beeinflussen.

    if ANALYSE_DATEI is None:

        return

    try:

        temporaer = Path(
            str(ANALYSE_DATEI) + ".tmp"
        )

        temporaer.write_text(
            json.dumps(ergebnisse, ensure_ascii=False, indent=2, default=str),
            encoding="utf-8"
        )

        os.replace(
            temporaer,
            ANALYSE_DATEI
        )

    except OSError as e:

        print(
            faerben(
                f"Marktlage nicht gespeichert: {type(e).__name__}",
                FARBE_GRAU
            )
        )


def analyse_laden():

    if ANALYSE_DATEI is None or not ANALYSE_DATEI.exists():

        return []

    try:

        daten = json.loads(
            ANALYSE_DATEI.read_text(encoding="utf-8")
        )

    except (OSError, ValueError):

        return []

    return daten if isinstance(daten, list) else []


# ============================================================
# 25. STATUSANZEIGE
# ============================================================

def status_anzeigen(konto, kurse):

    kapital = kontowert_berechnen(
        konto,
        kurse
    )

    gewinn = (
        kapital - konto["startkapital"]
    )

    rendite = (
        gewinn
        / konto["startkapital"]
        * 100
    )

    offenes_risiko = offenes_risiko_berechnen(konto)

    risiko_anteil = (
        offenes_risiko / kapital
        if kapital else 0.0
    )

    # --- Portfolio-Übersicht ---

    box_start("Portfolio", "💰")

    box_zeile(f"Gesamtvermögen     {usd(kapital)}")

    box_zeile(
        "Gewinn/Verlust     "
        + wert_faerben(gewinn, f"{usd(gewinn)}")
        + f"  ({rendite:+.2f} %)  {pfeil(gewinn)}"
    )

    box_zeile(f"Guthaben (frei)    {usd(konto['guthaben'])}")

    box_zeile(
        f"Offene Positionen  "
        f"{len(konto['positionen'])} / {MAX_OFFENE_POSITIONEN}"
    )

    box_zeile(
        f"Risiko genutzt     "
        f"{balken(risiko_anteil / MAX_GESAMTRISIKO)} "
        f"{risiko_anteil * 100:.2f} % / "
        f"{MAX_GESAMTRISIKO * 100:.0f} %"
    )

    box_ende()

    # --- Positionstabelle ---

    if konto["positionen"]:

        box_start("Offene Positionen", "📋")

        box_zeile(
            f"{'Coin':<10}{'Kurs':<11}{'Einstieg':<11}"
            f"{'P&L':<9}{'Stop-Loss':<11}Abstand"
        )

        for coin, position in konto["positionen"].items():

            kurs = kurse[coin]

            stop = position["stop_loss"]

            abstand_stop_prozent = (
                (kurs - stop) / kurs * 100
                if kurs else 0.0
            )

            unrealisiert_prozent = (
                (kurs - position["einstieg"])
                / position["einstieg"] * 100
            )

            box_zeile(
                spalte(COINS[coin]["name"], 10)
                + spalte(usd(kurs), 11)
                + spalte(usd(position["einstieg"]), 11)
                + spalte(
                    wert_faerben(
                        unrealisiert_prozent,
                        f"{unrealisiert_prozent:+.2f} %"
                    ),
                    9
                )
                + spalte(usd(stop), 11)
                + f"{abstand_stop_prozent:.2f} %"
            )

        box_ende()

        print(
            faerben(
                "ℹ️  Take-Profit: kein festes Ziel — "
                "Exit ausschließlich über Trailing-Stop",
                FARBE_GRAU
            )
        )


# ============================================================
# 26. AUTOMATISCHER FUNKTIONSTEST
# ============================================================

def smoke_test():

    unterkasten("Funktionstest", "🧪")

    # --- Test 1: Risk Gate / Positionsberechnung ---

    testkonto = neues_konto()

    testkurse = {

        "bitcoin": 70000.0,

        "ethereum": 2500.0,

        "solana": 120.0

    }

    test_atr = 1200.0

    pruefung, grund = risk_gate(
        testkonto,
        "bitcoin",
        testkurse,
        test_atr,
        spread=0.0005
    )

    if pruefung is None:

        raise RuntimeError(
            f"Risk-Gate-Test fehlgeschlagen: {grund}"
        )

    assert pruefung["menge"] > 0

    assert pruefung["stop_loss"] < pruefung["einstieg"]

    assert pruefung["geplantes_risiko"] <= (

        STARTKAPITAL
        * MAX_RISIKO_PRO_TRADE
        + 1e-8

    )

    print(faerben("  ✅ Risk Gate", FARBE_GRUEN))

    print(faerben("  ✅ Positionsberechnung", FARBE_GRUEN))

    # --- Test 2: ATR / ADX auf synthetischen Daten ---

    anzahl = 300

    index = pd.date_range(
        "2025-01-01",
        periods=anzahl,
        freq="h",
        tz="UTC"
    )

    # Als NumPy-Array erzeugen, nicht als pd.Series mit eigenem
    # RangeIndex - sonst richtet Pandas beim DataFrame-Bau nach
    # Index aus (RangeIndex vs. DatetimeIndex) und alles wird NaN.

    basis = (
        pd.RangeIndex(anzahl).to_numpy(dtype=float)
        * 5
        + 50000
    )

    test_df = pd.DataFrame({

        "open": basis,

        "high": basis + 80,

        "low": basis - 80,

        "close": basis + 20,

        "volume": 1.0

    }, index=index)

    test_atr_serie = atr_berechnen(
        test_df,
        ATR_PERIODE
    )

    test_adx_serie = adx_berechnen(
        test_df,
        ADX_PERIODE
    )

    letzter_atr = float(
        test_atr_serie.iloc[-1]
    )

    letzter_adx = float(
        test_adx_serie.iloc[-1]
    )

    assert gueltige_zahl(letzter_atr) and letzter_atr > 0

    assert gueltige_zahl(letzter_adx) and 0 <= letzter_adx <= 100

    print(faerben("  ✅ ATR-Berechnung", FARBE_GRUEN))

    print(faerben("  ✅ ADX-Berechnung", FARBE_GRUEN))

    # --- Test 3: Trailing-Stop zieht nur nach oben nach ---
    # Reine Funktion ohne Speichern - die echte Kontodatei
    # bleibt beim Funktionstest unberührt.

    testposition = {

        "menge": 0.01,

        "einstieg": 70000.0,

        "stop_loss": 67600.0,

        "atr_bei_einstieg": 1200.0,

        "hoechster_schlusskurs": 70000.0,

        "gesamtkosten": 700.0

    }

    hoechster, neuer_stop, geaendert = trailing_stop_berechnen(
        testposition,
        letzter_schluss=72000.0,
        aktueller_atr=1000.0
    )

    assert geaendert is True

    assert neuer_stop > 67600.0

    testposition["hoechster_schlusskurs"] = hoechster

    testposition["stop_loss"] = neuer_stop

    # Fallender Kurs darf den Stop nicht zurückziehen

    hoechster2, neuer_stop2, geaendert2 = trailing_stop_berechnen(
        testposition,
        letzter_schluss=69000.0,
        aktueller_atr=1000.0
    )

    assert geaendert2 is False

    assert neuer_stop2 == neuer_stop

    print(faerben("  ✅ Trailing-Stop-Logik", FARBE_GRUEN))

    # --- Test 4: Stop über dem Einstieg ist ein gültiger ---
    # Kontostand (V4.0 hat hier beim Speichern abgebrochen)

    testkonto = neues_konto()

    testposition = dict(testposition)

    testposition["stop_loss"] = testposition["einstieg"] + 500.0

    testkonto["positionen"]["bitcoin"] = testposition

    konto_pruefen(testkonto)

    assert offenes_risiko_berechnen(testkonto) == 0

    print(faerben("  ✅ Trailing-Stop über Einstieg", FARBE_GRUEN))


# ============================================================
# 27. COCKPIT-ANBINDUNG
# ============================================================

def neuer_lauf():

    # Zustand des laufenden Bots nur für die Anzeige. Die
    # Handelsentscheidungen hängen davon nicht ab.

    return {
        "konto": None,
        "kurse": None,
        "kurse_zeit_utc": None,
        "analyse": [],
        "gestartet_utc": zeitstempel(),
        "letzte_pruefung_utc": None,
        "naechste_pruefung_utc": None,
        "durchlaeufe": 0,
        "api_fehler": 0,
        "letzter_fehler": None,
        "cockpit_fehler": None,
        "alpaca_konto": ALPACA_KONTO,
        "kursfehler": KURSFEHLER,
        "spreads": SPREADS,
        "strategien": strategie_zustand_laden(),
        "statistik": blocker_statistik_laden(),
        "marktlage": {
            eintrag["coin"]: eintrag for eintrag in analyse_laden()
            if isinstance(eintrag, dict) and eintrag.get("coin") in COINS
        },
        "regime": {},
        "atr": {},
        # Leer = die stündliche Analyse läuft beim Start sofort
        "letzte_stunde": None,
        "letzte_viertelstunde": None
    }


def cockpit_aktualisieren(cockpit, lauf, stufe):

    # Das Cockpit liest nur, was dieser Durchlauf bereits
    # berechnet hat. Ein Fehler in der Anzeige darf den
    # Handel nie stoppen oder verändern: dann wird das
    # Cockpit für den Rest des Laufs abgeschaltet.

    if cockpit is None or lauf["cockpit_fehler"] is not None:

        return

    try:

        cockpit.update(
            lauf,
            stufe,
            list(EREIGNISSE)
        )

    except Exception as e:

        lauf["cockpit_fehler"] = type(e).__name__

        ereignis("COCKPIT_FEHLER", None, type(e).__name__)

        print(
            faerben(
                f"Cockpit konnte nicht aktualisiert werden "
                f"({type(e).__name__}) - der Handel läuft "
                "unverändert weiter.",
                FARBE_GELB
            )
        )


# ============================================================
# 28. EIN DURCHLAUF (alle 5 Minuten, auf volle 5 Minuten ausgerichtet)
# ============================================================

def ein_durchlauf(lauf):

    # Intern wird mit UTC gerechnet (Tageslimit-Reset,
    # Handelspause, Kerzengrenzen) - für die Anzeige
    # wird nur die Textausgabe nach Europe/Berlin
    # (MESZ/MEZ) umgerechnet.

    jetzt = datetime.now(
        timezone.utc
    )

    lauf["letzte_pruefung_utc"] = jetzt.isoformat()

    kasten(
        f"KURSPRÜFUNG · {lokale_zeit_anzeigen(jetzt)}",
        "🕒"
    )

    # ----------------------------------------
    # 1. Konto laden
    # ----------------------------------------

    konto = konto_laden()

    lauf["konto"] = konto

    # ----------------------------------------
    # 2. Aktuelle Alpaca-Kurse abrufen
    # ----------------------------------------

    kurse = aktuelle_kurse(konto)

    lauf["kurse"] = kurse

    lauf["kurse_zeit_utc"] = jetzt.isoformat()

    # ----------------------------------------
    # 3. Tagesstartwert festhalten
    # ----------------------------------------

    heute = jetzt.date().isoformat()

    if konto.get("tagesdatum") != heute:

        konto["tagesdatum"] = heute

        konto["tagesstartwert"] = (
            kontowert_berechnen(
                konto,
                kurse
            )
        )

        konto["tageslimit_erreicht"] = False

        konto_speichern(konto)

    # ----------------------------------------
    # 4. Bestehende Positionen überwachen
    # ----------------------------------------

    positionen_ueberwachen(
        konto,
        kurse
    )

    # ----------------------------------------
    # 5. Tagesverlustlimit prüfen
    # ----------------------------------------

    handel_erlaubt = tageslimit_pruefen(
        konto,
        kurse
    )

    # ----------------------------------------
    # 6. Marktlage jede volle Stunde und bei jedem Start
    #    (auch ohne Handelserlaubnis: Ausstiege und Stops)
    # ----------------------------------------

    aktuelle_stunde = jetzt.strftime("%Y-%m-%d-%H")

    if lauf["letzte_stunde"] != aktuelle_stunde:

        lauf["letzte_stunde"] = aktuelle_stunde

        konto["letzte_analyse"] = aktuelle_stunde

        konto_speichern(konto)

        stundenanalyse(konto, kurse, lauf)

    # ----------------------------------------
    # 7. Einstiegssignale alle 15 Minuten
    # ----------------------------------------

    viertelstunde = f"{aktuelle_stunde}-{jetzt.minute // 15}"

    if (
        handel_erlaubt
        and lauf["letzte_viertelstunde"] != viertelstunde
    ):

        lauf["letzte_viertelstunde"] = viertelstunde

        signalpruefung(konto, kurse, lauf)

    lauf["analyse"] = list(lauf["marktlage"].values())

    # ----------------------------------------
    # 8. Kontostand anzeigen
    # ----------------------------------------

    status_anzeigen(
        konto,
        kurse
    )

    print(
        faerben(
            "\nNächste Prüfung in ca. 5 Minuten.",
            FARBE_GRAU
        )
    )


# ============================================================
# 29. HAUPTSCHLEIFE
# ============================================================

def sekunden_bis_naechster_takt(jetzt=None):

    # Nächster Zeitpunkt HH:00/05/10/... + TAKT_VERZOEGERUNG
    # Sekunden - so liegen die 15-Minuten- und Stundenprüfungen
    # kurz nach dem jeweiligen Kerzenschluss.

    jetzt = jetzt or datetime.now(timezone.utc)

    sekunde = jetzt.timestamp()

    naechster = (sekunde // INTERVALL + 1) * INTERVALL + TAKT_VERZOEGERUNG

    if naechster - sekunde > INTERVALL:

        naechster -= INTERVALL

    return max(1.0, naechster - sekunde)


def hauptschleife(cockpit=None, durchlaeufe=None, warten=time.sleep):

    # durchlaeufe/warten nur für Tests; im Betrieb läuft die
    # Schleife, bis sie gestoppt wird.

    lauf = neuer_lauf()

    try:

        while True:

            try:

                ein_durchlauf(lauf)

                stufe = "WARTET"

            except DATEN_FEHLERTYPEN as e:

                print(
                    faerben(f"\n⚠️  DATEN-/API-FEHLER: {e}", FARBE_GELB)
                )

                print(
                    faerben(
                        "Keine Handelsausführung in diesem Durchlauf.",
                        FARBE_GELB
                    )
                )

                lauf["api_fehler"] += 1

                lauf["letzter_fehler"] = str(e)

                ereignis("API_FEHLER", None, str(e))

                stufe = "API_FEHLER"

            except Exception as e:

                kasten("KRITISCHER PROGRAMMFEHLER", "🔥")

                traceback.print_exc()

                print(
                    faerben(
                        "\nDer Bot wird aus Sicherheitsgründen "
                        "gestoppt.",
                        FARBE_ROT
                    )
                )

                lauf["letzter_fehler"] = (
                    f"{type(e).__name__}: {e}"
                )

                ereignis(
                    "PROGRAMMFEHLER",
                    None,
                    lauf["letzter_fehler"]
                )

                cockpit_aktualisieren(cockpit, lauf, "FEHLER")

                raise

            lauf["durchlaeufe"] += 1

            wartezeit = sekunden_bis_naechster_takt()

            lauf["naechste_pruefung_utc"] = (
                datetime.now(timezone.utc)
                + timedelta(seconds=wartezeit)
            ).isoformat()

            cockpit_aktualisieren(cockpit, lauf, stufe)

            if (
                durchlaeufe is not None
                and lauf["durchlaeufe"] >= durchlaeufe
            ):

                return lauf

            # Bis zur nächsten vollen 5 Minuten (+ Verzögerung) warten

            warten(wartezeit)

    except KeyboardInterrupt:

        kasten(f"TRADING BOT V{VERSION} GESTOPPT", "🛑")

        lauf["naechste_pruefung_utc"] = None

        cockpit_aktualisieren(cockpit, lauf, "GESTOPPT")

        print(
            faerben(
                "Das virtuelle Handelskonto bleibt in "
                "Google Drive gespeichert.",
                FARBE_GRAU
            )
        )

    return lauf


# ============================================================
# 29b. ALPACA-ZUGANG PRÜFEN (vor dem Start)
# ============================================================

# Die Startzelle liest die beiden Colab-Secrets und prüft die
# Schlüssel mit je einer lesenden Abfrage bei Alpaca. Jeder
# Fehler wird als verständlicher Hinweis gemeldet statt als
# Traceback - oder gar nicht.

SECRET_NAMEN = ("ALPACA_PAPER_API_KEY", "ALPACA_PAPER_SECRET_KEY")

SECRET_FEHLER = {
    "SecretNotFoundError": (
        "fehlt. In Colab links auf das Schlüsselsymbol (Secrets) "
        "klicken, ein Secret mit genau diesem Namen anlegen, den "
        "Alpaca-Paper-Schlüssel eintragen und den Schalter "
        "„Notebookzugriff“ einschalten."
    ),
    "NotebookAccessError": (
        "ist für dieses Notebook nicht freigegeben. In Colab links "
        "auf das Schlüsselsymbol (Secrets) klicken und beim Secret "
        "den Schalter „Notebookzugriff“ einschalten - oder die "
        "Freigabe-Abfrage mit „Zugriff gewähren“ bestätigen."
    ),
    "TimeoutException": (
        "wurde nicht freigegeben: Die Freigabe-Abfrage von Colab "
        "wurde nicht rechtzeitig bestätigt."
    )
}


class ZugangsFehler(RuntimeError):

    # Start nicht möglich: Secrets fehlen oder Alpaca lehnt die
    # Schlüssel ab. Der Text ist der Hinweis für die Anzeige.

    pass


def secrets_lesen(lesen):

    # lesen: google.colab.userdata.get. Gibt (key, secret) zurück.

    werte = []

    for name in SECRET_NAMEN:

        try:

            wert = lesen(name)

        except Exception as e:

            erklaerung = SECRET_FEHLER.get(
                type(e).__name__,
                f"konnte nicht gelesen werden ({type(e).__name__})."
            )

            raise ZugangsFehler(f"Secret {name} {erklaerung}") from None

        wert = str(wert or "").strip()

        if not wert:

            raise ZugangsFehler(
                f"Secret {name} ist leer. Unter Secrets den Schlüssel "
                "aus dem Alpaca-Dashboard (Bereich Paper Trading) "
                "eintragen."
            )

        werte.append(wert)

    return tuple(werte)


def zugang_fehlertext(fehler, was):

    status = getattr(fehler, "status_code", None)

    if status in (401, 403):

        return (
            f"Alpaca lehnt die Schlüssel ab (HTTP {status} bei {was}). "
            "Die Secrets enthalten keine gültigen Paper-Trading-"
            "Schlüssel: im Alpaca-Dashboard unter „Paper Trading“ "
            "neue API-Schlüssel erzeugen und beide Secrets ersetzen "
            "(Key ID und Secret Key gehören zusammen)."
        )

    if isinstance(fehler, requests.exceptions.RequestException):

        return (
            f"Alpaca ist nicht erreichbar ({type(fehler).__name__} bei "
            f"{was}). Die Zelle in ein paar Minuten erneut ausführen."
        )

    return f"Alpaca meldet einen Fehler bei {was}: {fehler}"


def alpaca_zugang_pruefen(broker, datenquelle):

    # Je eine lesende Abfrage: Paper-Konto und ein Kurs. Keine
    # Order. Gibt den Konto-Snapshot zurück.

    try:

        snapshot = konto_snapshot_lesen(broker)

    except (APIError, requests.exceptions.RequestException) as e:

        raise ZugangsFehler(zugang_fehlertext(e, "der Kontoabfrage")) from None

    try:

        datenquelle.get_crypto_latest_quote(
            CryptoLatestQuoteRequest(symbol_or_symbols=["BTC/USD"])
        )

    except (APIError, requests.exceptions.RequestException) as e:

        raise ZugangsFehler(zugang_fehlertext(e, "der Kursabfrage")) from None

    print(
        faerben(
            f"✅ Alpaca-Schlüssel gültig · Paper-Konto "
            f"{usd(snapshot['equity'])} (nur gelesen)",
            FARBE_GRUEN
        )
    )

    return snapshot


# ============================================================
# 30. START
# ============================================================

def start(
    datenquelle,
    broker=None,
    cockpit=None,
    ordner=STANDARD_ORDNER,
    drive_verbinden=True
):

    # datenquelle: alpaca CryptoHistoricalDataClient
    # broker: alpaca TradingClient (paper=True) - wird nur
    # einmal mit get_account() gelesen, nie für Orders genutzt.

    global HANDELSPAARE, DATENQUELLE, ALPACA_KONTO

    kasten(f"TRADING BOT V{VERSION}", "🤖")

    print(
        faerben(
            "Live-Demo · Alpaca-Daten · keine Orders".center(BREITE),
            FARBE_GRAU
        )
    )

    if drive_verbinden:

        from google.colab import drive

        drive.mount("/content/drive")

    speicherorte_einrichten(ordner)

    DATENQUELLE = datenquelle

    if broker is not None:

        ALPACA_KONTO = konto_snapshot_lesen(broker)

    HANDELSPAARE = handelspaare_ermitteln()

    konto = konto_uebernehmen()

    ohne_daten = [
        coin for coin in konto["positionen"]
        if coin not in HANDELSPAARE
    ]

    if ohne_daten:

        raise RuntimeError(
            "Offene Position ohne Alpaca-Kurs: "
            + ", ".join(ohne_daten)
        )

    if not PAPER_TRADING:

        raise RuntimeError(
            "Dieses Programm darf nur virtuell handeln."
        )

    # Zuerst alle Kernfunktionen überprüfen

    smoke_test()

    kasten(f"TRADING BOT V{VERSION} GESTARTET", "🚀")

    zeile("Handelsmodus", "PAPER TRADING", "📝")
    zeile("Datenquelle", "Alpaca (nur lesend, keine Orders)", "🔗")
    zeile("Kontowährung", "USD (virtuelles Budget)", "💵")
    zeile("Coins", f"{len(HANDELSPAARE)} von {len(COINS)}", "🪙")
    zeile(
        "Strategien",
        "A · Pullback, B · Squeeze (Parameter fest)",
        "📈"
    )
    zeile("Marktlage", "bei Start + jede volle Stunde", "🕐")
    zeile("Einstiegssignale", "alle 15 Minuten", "🎯")
    zeile("Positionsüberwachung", "alle 5 Minuten", "🔍")
    zeile(
        "Automatische Käufe",
        "aktiv" if AUTOMATISCHE_KAEUFE else "deaktiviert",
        "🤖"
    )
    zeile(
        "Cockpit",
        "aktiv" if cockpit is not None else "aus",
        "🖥️"
    )

    trenner()

    return hauptschleife(cockpit)


In [ ]:
%%writefile /content/crypto_cockpit_v63.py
"""Read-only cockpit for the crypto bot V6.3 (strategies A/B) on Alpaca live data (Colab view, browser tab, Drive file).

Shows only what the bot has already computed in the current pass (account, prices of the
last price check, last hourly market analysis, events). It never fetches market data and
never changes the account."""
from datetime import datetime, timedelta, timezone
from html import escape
from http.server import BaseHTTPRequestHandler, ThreadingHTTPServer
from pathlib import Path
import json
import math
import threading

import crypto_bot_v63 as bot


def safe(value):
    return escape(str(value if value is not None else '—'), quote=True)


def finite(value):
    return (isinstance(value, (int, float)) and not isinstance(value, bool)
            and math.isfinite(value))


def usd(value):
    return bot.usd(value) if finite(value) else '—'


def signed_usd(value):
    if not finite(value):
        return '—'
    return ('+' if value > 0 else '') + bot.usd(value)


def number(value, digits=2):
    if not finite(value):
        return '—'
    text = f'{value:,.{digits}f}'
    return text.replace(',', 'X').replace('.', ',').replace('X', '.')


def percent(value, digits=2, sign=True):
    if not finite(value):
        return '—'
    return (f'{value:+.{digits}f}' if sign else f'{value:.{digits}f}').replace('.', ',') + ' %'


def parse_utc(value):
    if value is None:
        return None
    moment = datetime.fromisoformat(value) if isinstance(value, str) else value
    if moment.tzinfo is None:
        moment = moment.replace(tzinfo=timezone.utc)
    return moment.astimezone(timezone.utc)


def berlin(value, with_date=True):
    """Short local time for tables; the text log uses bot.lokale_zeit_anzeigen()."""
    moment = parse_utc(value)
    if moment is None:
        return '—'
    local = moment.astimezone(bot.BERLIN_ZEITZONE)
    return local.strftime('%d.%m. %H:%M' if with_date else '%H:%M')


def tone(value):
    if not finite(value) or value == 0:
        return 'neutral'
    return 'ok' if value > 0 else 'bad'


def table(headings, rows, empty):
    header = ''.join(f'<th>{safe(h)}</th>' for h in headings)
    body = ''.join('<tr>' + ''.join(f'<td>{cell}</td>' for cell in row) + '</tr>'
                   for row in rows)
    if not body:
        body = f'<tr><td colspan="{len(headings)}" class="muted">{safe(empty)}</td></tr>'
    return f'<div class="scroll"><table><thead><tr>{header}</tr></thead><tbody>{body}</tbody></table></div>'


# ---------- snapshot: numbers from the bot's own state, no market data ----------

def liquidation_value(menge, kurs):
    """Net proceeds of a virtual sale at `kurs`, with the bot's simulated costs."""
    return menge * kurs * (1 - bot.SLIPPAGE) * (1 - bot.GEBUEHR)


def closed_trades(transaktionen, limit=12):
    """Pair each VERKAUF with the preceding KAUF of the same coin."""
    offene_kaeufe = {}
    trades = []
    for t in transaktionen:
        if t.get('aktion') == 'KAUF':
            offene_kaeufe[t.get('coin')] = t
        elif t.get('aktion') == 'VERKAUF':
            kauf = offene_kaeufe.pop(t.get('coin'), {})
            trades.append(dict(coin=t.get('coin'), menge=t.get('menge'),
                               kaufzeit=kauf.get('zeit'), verkaufszeit=t.get('zeit'),
                               kaufkurs=kauf.get('kurs'), verkaufskurs=t.get('kurs'),
                               gewinn=t.get('gewinn'), grund=t.get('grund')))
    return trades[-limit:], len(trades)


def trading_pause_until(konto, coin, now):
    letzter = konto.get('letzter_ausstieg', {}).get(coin)
    if not letzter:
        return None
    ende = parse_utc(letzter) + timedelta(seconds=bot.HANDELSPAUSE)
    return ende.isoformat() if ende > now else None


def snapshot(lauf, stufe, ereignisse, now=None):
    now = parse_utc(now) or datetime.now(timezone.utc)
    konto, kurse = lauf.get('konto'), lauf.get('kurse')
    data = dict(version=bot.VERSION, stufe=stufe, paper_trading=bot.PAPER_TRADING,
                echte_orders=0, aktualisiert_utc=now.isoformat(),
                gestartet_utc=lauf.get('gestartet_utc'),
                letzte_pruefung_utc=lauf.get('letzte_pruefung_utc'),
                naechste_pruefung_utc=lauf.get('naechste_pruefung_utc'),
                kurse_zeit_utc=lauf.get('kurse_zeit_utc'),
                durchlaeufe=lauf.get('durchlaeufe', 0),
                api_fehler=lauf.get('api_fehler', 0),
                letzter_fehler=lauf.get('letzter_fehler'),
                intervall_sekunden=bot.INTERVALL,
                max_positionen=bot.MAX_OFFENE_POSITIONEN,
                max_gesamtrisiko=bot.MAX_GESAMTRISIKO,
                max_tagesverlust=bot.MAX_TAGESVERLUST,
                bewertet=False, positionen=[], abschluesse=[], abschluesse_gesamt=0, auswertung={},
                blocker=lauf.get('statistik') or {},
                analyse=[], ereignisse=[dict(e) for e in list(ereignisse)[-10:]],
                alpaca_konto=lauf.get('alpaca_konto'),
                kursfehler={bot.COINS[c]['name']: grund
                            for c, grund in (lauf.get('kursfehler') or {}).items()},
                coins_aktiv=len(bot.HANDELSPAARE), coins_gesamt=len(bot.COINS))
    if konto is None or kurse is None or any(c not in kurse for c in konto['positionen']):
        # Before the first successful price check there is nothing to value.
        return data

    kapital = bot.kontowert_berechnen(konto, kurse)
    offenes_risiko = bot.offenes_risiko_berechnen(konto)
    positionen = []
    for coin, p in konto['positionen'].items():
        kurs = kurse[coin]
        netto = liquidation_value(p['menge'], kurs)
        stop_erloes = liquidation_value(p['menge'], p['stop_loss'])
        positionen.append(dict(
            coin=coin, name=bot.COINS[coin]['name'], menge=p['menge'],
            einstieg=p['einstieg'], kaufzeit=p.get('kaufzeit'), kurs=kurs,
            marktwert=p['menge'] * kurs, gesamtkosten=p['gesamtkosten'],
            ergebnis_nach_kosten=netto - p['gesamtkosten'],
            kursaenderung_prozent=(kurs / p['einstieg'] - 1) * 100,
            stop=p['stop_loss'], stop_zu_einstieg=p['stop_loss'] - p['einstieg'],
            stop_zu_einstieg_prozent=(p['stop_loss'] / p['einstieg'] - 1) * 100,
            abstand_kurs_stop_prozent=(kurs - p['stop_loss']) / kurs * 100,
            ergebnis_am_stop=stop_erloes - p['gesamtkosten'],
            hoechster_schlusskurs=p['hoechster_schlusskurs'],
            atr_bei_einstieg=p['atr_bei_einstieg'],
            strategie=p.get('strategie'),
            stop_erreicht=kurs <= p['stop_loss']))

    verkaeufe = [t for t in konto['transaktionen'] if t.get('aktion') == 'VERKAUF']
    abschluesse, anzahl = closed_trades(konto['transaktionen'])
    tagesstart = konto.get('tagesstartwert')
    analyse = []
    for eintrag in lauf.get('analyse') or []:
        eintrag = dict(eintrag)
        coin = eintrag.get('coin')
        eintrag['name'] = bot.COINS.get(coin, {}).get('name', coin)
        eintrag['handelspause_bis'] = trading_pause_until(konto, coin, now)
        eintrag['kurs'] = kurse.get(coin)
        eintrag['spread'] = (lauf.get('spreads') or {}).get(coin)
        analyse.append(eintrag)

    data.update(
        bewertet=True,
        kapital=kapital, startkapital=konto['startkapital'],
        gewinn_gesamt=kapital - konto['startkapital'],
        rendite_prozent=(kapital / konto['startkapital'] - 1) * 100,
        guthaben=konto['guthaben'],
        marktwert_positionen=sum(p['marktwert'] for p in positionen),
        offenes_ergebnis=sum(p['ergebnis_nach_kosten'] for p in positionen),
        realisiert=sum(t.get('gewinn', 0) for t in verkaeufe),
        gebuehren=sum(t.get('gebuehr', 0) for t in konto['transaktionen']),
        offenes_risiko=offenes_risiko,
        risiko_anteil=offenes_risiko / kapital if kapital else 0.0,
        tagesdatum=konto.get('tagesdatum'), tagesstartwert=tagesstart,
        tagesergebnis=kapital - tagesstart if finite(tagesstart) else None,
        tagesergebnis_prozent=(kapital / tagesstart - 1) * 100 if finite(tagesstart) and tagesstart else None,
        tagesgrenze=tagesstart * (1 - bot.MAX_TAGESVERLUST) if finite(tagesstart) else None,
        tageslimit_erreicht=bool(konto.get('tageslimit_erreicht')),
        positionen=positionen, abschluesse=abschluesse, abschluesse_gesamt=anzahl,
        auswertung=bot.strategie_auswertung(konto['transaktionen']),
        analyse=analyse)
    return data


# ---------- HTML ----------

STATUS_TEXT = {
    'WARTET': ('Bot läuft · wartet auf die nächste Kursprüfung', 'ok'),
    'API_FEHLER': ('Alpaca-Daten nicht verfügbar · kein Handel in diesem Durchlauf, nächster Versuch folgt', 'bad'),
    'GESTOPPT': ('Bot gestoppt', 'neutral'),
    'FEHLER': ('Programmfehler · Bot aus Sicherheitsgründen gestoppt', 'bad'),
}
# No price check is expected in these stages: an old timestamp is not a stall.
QUIET_STAGES = ('GESTOPPT', 'FEHLER')


BLOCKER_LABELS = (('daten', 'Datenprüfung · stündlich je Coin'),
                  ('a', 'A · Pullback-Erkennung · stündlich'),
                  ('b', 'B · Squeeze-Erkennung · stündlich'),
                  ('signal_a', 'A · 15-Min-Signal'),
                  ('signal_b', 'B · 15-Min-Signal'),
                  ('risk_gate', 'Risk Gate'))

REGIME = {'AUFWAERTSTREND': ('Aufwärtstrend', 'ok'), 'ABWAERTSTREND': ('Abwärtstrend', 'bad'),
          'NEUTRAL': ('neutral', 'muted')}


def strategy_name(key):
    return safe(bot.STRATEGIE_NAMEN.get(key, '—'))


def regime_cell(eintrag):
    if eintrag.get('status') == 'DATENFEHLER':
        return f'<span class="bad">Datenfehler</span><br><small>{safe(eintrag.get("fehler"))}</small>'
    text, css = REGIME.get(eintrag.get('regime'), ('—', 'muted'))
    return f'<span class="{css}">{text}</span>'


def strategy_a_cell(a):
    status = (a or {}).get('status')
    if status == 'BEREIT':
        return (f'<span class="ok">Pullback bereit</span><br><small>Tief {usd(a.get("tief"))} · '
                f'bis {berlin(a.get("bis"), False)}</small>')
    if status in ('BENUTZT', 'ABGELAUFEN'):
        return f'<span class="neutral">{safe(status.lower())}</span><br><small>{safe(a.get("grund"))}</small>'
    return f'<span class="muted">inaktiv</span><br><small>{safe((a or {}).get("grund"))}</small>'


def strategy_b_cell(b):
    b = b or {}
    status = b.get('status')
    width = ''
    if finite(b.get('bbw')) and finite(b.get('schwelle')):
        width = f'BBW {number(b["bbw"] * 100, 2)} % / Schwelle {number(b["schwelle"] * 100, 2)} %'
    if status == 'AKTIV':
        return (f'<span class="ok">Squeeze aktiv</span><br><small>Spanne {usd(b.get("unten"))} – '
                f'{usd(b.get("oben"))} · bis {berlin(b.get("bis"), False)}<br>{width}</small>')
    if status in ('BENUTZT', 'ABGELAUFEN'):
        return f'<span class="neutral">{safe(status.lower())}</span><br><small>{width}</small>'
    return f'<span class="muted">kein Squeeze</span><br><small>{safe(b.get("grund"))}{" · " + width if width else ""}</small>'


def render_html(data):
    """One self-contained HTML surface; every text from outside is escaped."""
    status, status_class = STATUS_TEXT.get(data['stufe'], (data['stufe'], 'neutral'))
    if data['bewertet'] and data['tageslimit_erreicht']:
        entry_status = ('bad', 'Neue Käufe gesperrt: Tagesverlustlimit erreicht (bis Tageswechsel UTC)')
    elif data['stufe'] in QUIET_STAGES:
        entry_status = ('neutral', 'Keine neuen Käufe: Bot läuft nicht')
    else:
        entry_status = ('neutral', 'Neue Käufe: 15-Min-Signal (A oder B) und Risk Gate')

    def tile(label, value, sub=''):
        sub = f'<small>{sub}</small>' if sub else ''
        return f'<div class="tile"><span>{safe(label)}</span><strong>{value}</strong>{sub}</div>'

    if data['bewertet']:
        risk_share = data['risiko_anteil'] / data['max_gesamtrisiko'] if data['max_gesamtrisiko'] else 0
        bar = (f'<div class="bar"><i style="width:{max(0, min(100, risk_share * 100)):.0f}%"'
               f' class="{"bad" if risk_share >= 0.8 else ""}"></i></div>')
        day_sub = (f'Grenze {usd(data["tagesgrenze"])} (−{percent(data["max_tagesverlust"] * 100, 0, False)})'
                   if data['tagesgrenze'] is not None else '')
        cards = ''.join((
            tile('Gesamtvermögen · Modell', usd(data['kapital']),
                 f'Start {usd(data["startkapital"])}'),
            tile('Gewinn/Verlust seit Start',
                 f'<span class="{tone(data["gewinn_gesamt"])}">{signed_usd(data["gewinn_gesamt"])}</span>',
                 percent(data['rendite_prozent'])),
            tile('Heute (UTC-Tag)',
                 f'<span class="{tone(data["tagesergebnis"])}">{signed_usd(data["tagesergebnis"])}</span>',
                 day_sub),
            tile('Freies Guthaben', usd(data['guthaben'])),
            tile('In Positionen · Marktwert', usd(data['marktwert_positionen']),
                 f'offen nach Kosten {signed_usd(data["offenes_ergebnis"])}'),
            tile('Offenes Risiko bis Stop', usd(data['offenes_risiko']),
                 f'{percent(data["risiko_anteil"] * 100, 2, False)} von '
                 f'{percent(data["max_gesamtrisiko"] * 100, 0, False)}{bar}'),
            tile('Realisiert', f'<span class="{tone(data["realisiert"])}">{signed_usd(data["realisiert"])}</span>',
                 f'Gebühren gesamt {usd(data["gebuehren"])}'),
            tile('Positionen offen / max · Trades',
                 f'{len(data["positionen"])} / {data["max_positionen"]}',
                 f'{data["abschluesse_gesamt"]} abgeschlossen'),
        ))
    else:
        cards = tile('Konto', 'noch nicht bewertet', 'Warte auf die erste erfolgreiche Kursprüfung')

    pos_rows = []
    for p in data['positionen']:
        stop_tone = tone(p['stop_zu_einstieg'])
        at_stop = p['ergebnis_am_stop']
        stop_note = (f'sichert {signed_usd(at_stop)} nach Kosten' if at_stop > 0
                     else f'Risiko bis Stop {usd(-at_stop)}')
        state = ('<span class="bad">Stop erreicht</span>' if p['stop_erreicht']
                 else f'{percent(p["abstand_kurs_stop_prozent"], 2, False)} über Stop')
        pos_rows.append((
            f'<b>{safe(p["name"])}</b><br><small>{strategy_name(p["strategie"])} · '
            f'{number(p["menge"], 6)}</small>',
            f'{usd(p["einstieg"])}<br><small>{berlin(p["kaufzeit"])}</small>',
            f'{usd(p["kurs"])}<br><small>{state}</small>',
            f'<span class="{tone(p["ergebnis_nach_kosten"])}">{signed_usd(p["ergebnis_nach_kosten"])}</span>'
            f'<br><small>Kurs {percent(p["kursaenderung_prozent"])} ggü. Einstieg</small>',
            f'<span class="{stop_tone}">{usd(p["stop"])} ({percent(p["stop_zu_einstieg_prozent"])})</span>'
            f'<br><small>{stop_note}</small>',
            f'{usd(p["hoechster_schlusskurs"])}<br><small>ATR Einstieg {usd(p["atr_bei_einstieg"])}</small>'))

    grund_text = bot.GRUND_TEXTE
    trade_rows = [(
        safe(bot.COINS.get(t['coin'], {}).get('name', t['coin']))
        + f'<br><small>{strategy_name(t.get("strategie"))}</small>',
        f'{berlin(t["kaufzeit"])} → {berlin(t["verkaufszeit"])}',
        number(t['menge'], 6), usd(t['kaufkurs']), usd(t['verkaufskurs']),
        f'<span class="{tone(t["gewinn"])}">{signed_usd(t["gewinn"])}</span>',
        safe(grund_text.get(t['grund'], t['grund'])))
        for t in reversed(data['abschluesse'])]

    analysis_rows = []
    for a in data['analyse']:
        adx = a.get('adx')
        adx_text = number(adx, 1)
        if finite(adx):
            adx_text = f'<span class="{"ok" if adx >= bot.ADX_MINDESTWERT else "muted"}">{adx_text}</span>'
        quiet = (f' · {int(a["ohne_handel_50h"])}/50 h ohne Handel'
                 if finite(a.get('ohne_handel_50h')) and a['ohne_handel_50h'] else '')
        spread = f'Spread {percent(a["spread"] * 100, 3, False)}' if finite(a.get('spread')) else ''
        signal = a.get('signal') or {}
        decision = safe(signal.get('text') or '—')
        if signal.get('zeit'):
            decision = f'{decision}<br><small>{berlin(signal["zeit"], False)}</small>'
        pause = a.get('handelspause_bis')
        if pause:
            decision += f'<br><small>Handelspause bis {berlin(pause)}</small>'
        analysis_rows.append((
            f'<b>{safe(a.get("name"))}</b><br><small>Kerze {berlin(a.get("kerze_utc"))}{quiet}</small>',
            f'{usd(a.get("kurs"))}<br><small>{spread}</small>',
            regime_cell(a),
            f'{usd(a.get("ema_kurz"))}<br>{usd(a.get("ema_mittel"))}<br>{usd(a.get("ema_lang"))}',
            f'{adx_text}<br><small>ATR {usd(a.get("atr"))}</small>',
            strategy_a_cell(a.get('a')), strategy_b_cell(a.get('b')), decision))

    stats_rows = []
    for key, s in data['auswertung'].items():
        r_avg = s['r_durchschnitt']
        r_text = (f'<span class="{tone(r_avg)}">{number(r_avg, 2)} R</span>'
                  f'<br><small>aus {s["r_anzahl"]} Trades</small>' if r_avg is not None else '—')
        reasons = ', '.join(f'{safe(grund_text.get(g, g))}: {n}'
                            for g, n in sorted(s['gruende'].items(), key=lambda x: (-x[1], x[0])))
        stats_rows.append((
            f'<b>{strategy_name(key)}</b>',
            str(s['trades']),
            f'{percent(s["trefferquote"] * 100, 0, False)}<br><small>{s["gewinner"]} Gewinner</small>',
            f'<span class="{tone(s["ergebnis"])}">{signed_usd(s["ergebnis"])}</span>'
            f'<br><small>Ø {signed_usd(s["durchschnitt"])} je Trade</small>',
            r_text, reasons))

    blocker_rows = []
    blocker = data.get('blocker') or {}
    for stufe, label in BLOCKER_LABELS:
        counts = blocker.get(stufe) or {}
        total = sum(counts.values())
        if not total:
            continue
        top = sorted(counts.items(), key=lambda x: (-x[1], x[0]))[:6]
        blocker_rows.append((
            f'<b>{safe(label)}</b>', str(total),
            '<br>'.join(f'{safe(g)}: {n} ({percent(n / total * 100, 0, False)})' for g, n in top)))

    link_note = ''
    if data.get('link'):
        link_note = (f'<p>Eigener Browser-Tab: <a href="{safe(data["link"])}" target="{safe(TAB_NAME)}">'
                     f'{safe(LINK_TEXT)}</a></p>')

    event_rows = [(berlin(e.get('zeit')),
                   safe(bot.COINS.get(e.get('coin'), {}).get('name', e.get('coin'))),
                   safe(e.get('art')), safe(e.get('text')))
                  for e in reversed(data['ereignisse'])]

    analysis_time = next((a.get('zeit') for a in data['analyse'] if a.get('zeit')), None)
    error_note = ''
    if data['letzter_fehler'] and data['stufe'] in ('API_FEHLER', 'FEHLER'):
        error_note = f'<br><span class="bad">Letzter Fehler: {safe(data["letzter_fehler"])}</span>'
    trend_note = ('Marktlage vorläufig definiert (Abschnitte 4.1/4.3 fehlen): Aufwärtstrend = EMA 50 über EMA 200 und ADX ≥ '
                  f'{bot.ADX_MINDESTWERT}; Abwärtstrend = EMA 50 unter EMA 200 und Schluss unter EMA 200. '
                  'Alle Parameter sind feste Testparameter; der Bot verändert sie nie selbst.'
                  if bot.TREND_DEFINITION_VORLAEUFIG else '')
    missing_note = ''
    if data['kursfehler']:
        missing_note = ('<br><span class="bad">Ohne aktuellen Kurs: ' +
                        safe('; '.join(f'{k}: {v}' for k, v in data['kursfehler'].items())) + '</span>')
    snapshot_equity = (data['alpaca_konto'] or {}).get('equity')
    account_note = (f'Start-Snapshot des Alpaca-Paper-Kontos: {usd(snapshot_equity)}; '
                    'niemals mit dem virtuellen Bot-Konto verrechnen.' if snapshot_equity is not None
                    else 'Alpaca-Paper-Konto nicht abgefragt.')
    updated_ms = int(parse_utc(data['aktualisiert_utc']).timestamp() * 1000)
    stall_minutes = 2 * data['intervall_sekunden'] // 60 + 1
    return f'''<!doctype html><html lang="de"><head><meta charset="utf-8"><style>
.kc{{font:14px/1.42 system-ui,-apple-system,sans-serif;color:#182338;background:#f5f8fc;
padding:18px;border-radius:14px;max-width:1260px;margin:0 auto}}
.kc h2{{margin:0 0 6px;font-size:22px}} .kc h3{{font-size:16px;margin:18px 0 7px}}
.kc .subtitle,.kc small,.kc .muted{{color:#58677a}}
.kc .ok{{color:#177245;font-weight:700}} .kc .bad{{color:#ba3042;font-weight:700}}
.kc .neutral{{color:#344e75;font-weight:700}}
.kc .tiles{{display:grid;grid-template-columns:repeat(auto-fit,minmax(155px,1fr));gap:9px;margin:14px 0}}
.kc .tile{{background:white;border:1px solid #d9e1ed;border-radius:10px;padding:10px 12px}}
.kc .tile>span{{display:block;color:#58677a;font-size:12px}} .kc .tile strong{{font-size:17px;display:block}}
.kc .bar{{height:6px;background:#e1e7f0;border-radius:3px;margin-top:5px;overflow:hidden}}
.kc .bar i{{display:block;height:100%;background:#177245}} .kc .bar i.bad{{background:#ba3042}}
.kc .scroll{{overflow-x:auto}} .kc table{{border-collapse:collapse;width:100%;background:white}}
.kc th,.kc td{{border-bottom:1px solid #e1e7f0;text-align:left;padding:8px 9px;vertical-align:top;white-space:nowrap}}
.kc td:last-child{{white-space:normal;min-width:190px}}
.kc th{{background:#eaf0fa;font-size:12px}} .kc .note{{margin-top:13px;font-size:12px;color:#47566d}}
</style></head><body><div class="kc">
<h2>Krypto-Bot V{safe(data['version'])} · Live-Demo · Strategien A + B</h2>
<div class="subtitle" id="kc-stand" data-updated-ms="{updated_ms}" data-quiet="{'1' if data['stufe'] in QUIET_STAGES else '0'}" data-stall-min="{stall_minutes}">Stand {berlin(data['aktualisiert_utc'])} ·
Kurse vom {berlin(data['kurse_zeit_utc'])} · nächste Prüfung ca. {berlin(data['naechste_pruefung_utc'], False)} ·
letzte Marktlage {berlin(analysis_time)} · Zeiten Europe/Berlin</div>
<p><span class="{status_class}">{safe(status)}</span> · Alpaca-Daten · {data['coins_aktiv']} von {data['coins_gesamt']} Coins in USD ·
Alpaca-Orders: 0 · alle Trades ausschließlich virtuell · {data['durchlaeufe']} Prüfungen, {data['api_fehler']} Daten-/API-Fehler seit Start{error_note}{missing_note}<br>
<span class="{entry_status[0]}">{safe(entry_status[1])}</span></p>
{link_note}<div class="tiles">{cards}</div>
<h3>Offene virtuelle Positionen</h3>
{table(('Coin / Menge', 'Einstieg', 'Kurs · Abstand', 'Ergebnis nach Kosten', 'Trailing-Stop', 'Höchster Schluss'), pos_rows, 'Keine offene Position.')}
<h3>Marktlage und Strategien (Marktlage stündlich, Signale alle 15 Min.)</h3>
{table(('Coin', 'Kurs · Spread', 'Marktlage', f'EMA {bot.EMA_KURZ} / {bot.EMA_MITTEL} / {bot.EMA_LANG}', f'ADX (≥ {bot.ADX_MINDESTWERT}) · ATR', 'A · Pullback', 'B · Squeeze', 'Letzte 15-Min-Prüfung'), analysis_rows, 'Noch keine Marktlage gespeichert.')}
<p class="note">{safe(trend_note)}</p>
<h3>Abgeschlossene virtuelle Trades · letzte 12</h3>
{table(('Coin', 'Kauf → Verkauf', 'Menge', 'Kaufkurs', 'Verkaufskurs', 'Ergebnis', 'Ausstieg'), trade_rows, 'Noch kein abgeschlossener Trade.')}
<h3>Auswertung je Strategie · alle abgeschlossenen Trades</h3>
{table(('Strategie', 'Trades', 'Trefferquote', 'Ergebnis nach Kosten', 'Ø Ergebnis in R', 'Ausstiege'), stats_rows, 'Noch kein abgeschlossener Trade.')}
<p class="note">R = Ergebnis geteilt durch das Risiko bis zum Anfangs-Stop inkl. Kosten (1 R = höchstens 1 % des Kapitals beim Kauf).
Aussagekräftig erst ab etwa 30 Trades je Strategie; vorher nicht an den Parametern drehen.</p>
<h3>Warum kein Trade? · Prüfungen seit {berlin(blocker.get('seit'))}</h3>
{table(('Prüfstufe', 'Prüfungen', 'Ergebnis (häufigste zuerst)'), blocker_rows, 'Noch keine Prüfung gezählt.')}
<p class="note">Stündlich je Coin: Daten, A (Pullback) und B (Squeeze); alle 15 Min. je bereitem Coin: Einstiegssignal; danach das Risk Gate.
Zeigt, welche Bedingung Einstiege am häufigsten verhindert - Grundlage für die nächste Überprüfung der Parameter.</p>
<h3>Letzte Ereignisse (seit Start dieser Sitzung)</h3>
{table(('Zeit', 'Coin', 'Ereignis', 'Details'), event_rows, 'Noch keine Ereignisse.')}
<div class="note">Virtuelles Paper Trading: Alpaca wird nur gelesen, es gehen keine Orders an Alpaca, virtuelle Trades erscheinen nicht im Alpaca-Konto.
Kurs = Mittelkurs aus Alpaca-Bid/Ask zum Zeitpunkt der Kursprüfung (alle 5 Minuten). Stunden ohne Handel sind als flache Kerze aufgefüllt. Ergebnisse enthalten die simulierten Kosten (Gebühr {percent(bot.GEBUEHR * 100, 2, False)},
Slippage {percent(bot.SLIPPAGE * 100, 1, False)} je Seite). Der Stop wird alle 5 Minuten gegen diesen Kurs geprüft und stündlich mit
{number(bot.ATR_TRAILING_MULTIPLIKATOR, 1)}×ATR nachgezogen; Kursbewegungen zwischen zwei Prüfungen sieht der Bot nicht.
Zusätzliche Ausstiege: A bei Stundenschluss unter EMA {bot.EMA_MITTEL}; B bei Stundenschluss zurück in der Spanne oder nach
{bot.B_ZEITSTOP_STUNDEN} h ohne +1×ATR.
Stop rot = unter Einstieg, grün = darüber. Tagesverlustlimit und Tageswechsel nach UTC.<br>{account_note}</div>
</div></body></html>'''


# ---------- browser tab and Drive file ----------

BROWSER_REFRESH_SECONDS = 15


def browser_page(page, refresh_seconds=BROWSER_REFRESH_SECONDS):
    """Same view for a normal browser tab: reloads itself and shows its age."""
    head = (f'<meta charset="utf-8"><meta http-equiv="refresh" content="{int(refresh_seconds)}">'
            '<meta name="viewport" content="width=device-width,initial-scale=1">')
    age = ('<p class="muted" id="kc-age"></p><script>(function(){'
           'var s=document.getElementById("kc-stand");if(!s)return;'
           'var e=document.getElementById("kc-age");'
           'var m=Math.floor((Date.now()-Number(s.dataset.updatedMs))/60000);if(!(m>=0))return;'
           f'e.textContent="Browseransicht lädt alle {int(refresh_seconds)} s neu · Stand ist "+m+" Min. alt";'
           'if(m>=Number(s.dataset.stallMin)&&s.dataset.quiet!=="1"){e.className="bad";'
           'e.textContent+=" – Bot läuft evtl. nicht mehr (Colab prüfen)";}})();</script>')
    return page.replace('<meta charset="utf-8">', head, 1).replace(
        '<div class="tiles">', age + '<div class="tiles">', 1)


WAITING_PAGE = ('<!doctype html><html lang="de"><head><meta charset="utf-8"></head>'
                '<body style="font:15px system-ui,sans-serif;padding:24px;background:#f5f8fc;color:#182338">'
                '<h2>Krypto-Bot V6.3 · Live-Demo</h2><p>Warte auf die erste Kursprüfung …</p>'
                '<div class="tiles"></div></body></html>')


def message_page(title, text):
    """Plain page with one red notice (start problems); reloads like the cockpit tab."""
    return ('<!doctype html><html lang="de"><head><meta charset="utf-8">'
            f'<meta http-equiv="refresh" content="{BROWSER_REFRESH_SECONDS}"></head>'
            '<body style="font:15px system-ui,sans-serif;padding:24px;background:#f5f8fc;color:#182338">'
            f'<h2>Krypto-Bot V{safe(bot.VERSION)} · Live-Demo</h2>'
            f'{notice_html(title, text)}</body></html>')


def notice_html(title, text, ok=False):
    colours = ('#1f6f3a', '#eaf7ee', '#9ad3ad') if ok else ('#8a1c2b', '#fdecee', '#e0707e')
    return (f'<div style="font:15px system-ui,sans-serif;margin:8px 0;padding:12px 14px;border-radius:8px;'
            f'color:{colours[0]};background:{colours[1]};border:2px solid {colours[2]}">'
            f'<b>{safe(title)}</b><br>{safe(text)}</div>')


def startfehler_anzeigen(fehler, cockpit=None):
    """Show why the bot did not start: red box in the start cell, and in the cockpit if it exists."""
    title = 'Bot nicht gestartet: Alpaca-Zugang'
    print(f'\033[91m❌ {title}\n{fehler}\033[0m')
    try:
        from IPython.display import HTML, display
        display(HTML(notice_html(title, str(fehler))))
    except ImportError:
        pass
    if cockpit is not None:
        cockpit.show_message(title, str(fehler))


def write_atomic(path, payload):
    temp = path.with_name(path.name + '.incomplete')
    temp.write_text(payload, encoding='utf-8')
    temp.replace(path)


class Cockpit:
    def __init__(self, display_enabled=True, latest_path=None):
        """`latest_path`: one fixed, self-refreshing HTML file (e.g. on Drive)
        that always shows the newest view."""
        self.display_enabled = display_enabled
        self.latest_path = Path(latest_path) if latest_path is not None else None
        self.latest_browser_page = browser_page(WAITING_PAGE)
        self.write_errors = 0
        self.handle = None
        self.link = None          # browser-tab link, once known (Colab)

    def _show(self, page):
        if not self.display_enabled:
            return
        from IPython.display import HTML, display
        if self.handle is None:
            self.handle = display(HTML(page), display_id=True)
        else:
            self.handle.update(HTML(page))

    def show_placeholder(self):
        """Reserve the cockpit's place at the top of the cell output before the text log starts."""
        self._show(WAITING_PAGE)

    def show_message(self, title, text):
        """A start problem (e.g. Alpaca keys) in the cell view, the browser tab and the Drive file."""
        page = message_page(title, text)
        self.latest_browser_page = page
        if self.latest_path is not None:
            try:
                self.latest_path.parent.mkdir(parents=True, exist_ok=True)
                write_atomic(self.latest_path, page)
            except OSError:
                self.write_errors += 1
        self._show(page)

    def update(self, lauf, stufe, ereignisse=(), now=None):
        data = snapshot(lauf, stufe, ereignisse, now)
        data['link'] = self.link
        page = render_html(data)
        self.latest_browser_page = browser_page(page)   # served by the live link
        if self.latest_path is not None:
            try:
                self.latest_path.parent.mkdir(parents=True, exist_ok=True)
                write_atomic(self.latest_path, self.latest_browser_page)
            except OSError as exc:
                # A Drive hiccup must not switch the cockpit off; the next update writes again.
                self.write_errors += 1
                print('Cockpit-Datei nicht geschrieben:', self.latest_path.name, type(exc).__name__)
        self._show(page)
        return data


# ---------- read-only web server for the browser tab ----------

class _Handler(BaseHTTPRequestHandler):
    cockpit = None

    def do_GET(self):
        if self.path.split('?', 1)[0] not in ('/', '/index.html'):
            self.send_error(404)
            return
        payload = self.cockpit.latest_browser_page.encode('utf-8')
        self.send_response(200)
        self.send_header('Content-Type', 'text/html; charset=utf-8')
        self.send_header('Cache-Control', 'no-store')
        self.send_header('Content-Length', str(len(payload)))
        self.end_headers()
        self.wfile.write(payload)

    def log_message(self, *args):   # keep the notebook output clean
        pass


def serve_cockpit(cockpit, port=8771, attempts=20):
    """Serve the newest cockpit page over HTTP from a daemon thread.

    Read-only: only the page already rendered by Cockpit.update is returned;
    no market data, no account changes, no file system access."""
    handler = type('CockpitHandler', (_Handler,), dict(cockpit=cockpit))
    for candidate in range(port, port + attempts):
        try:
            server = ThreadingHTTPServer(('', candidate), handler)
        except OSError:
            continue   # e.g. a previous run of the cell still holds the port
        server.daemon_threads = True
        threading.Thread(target=server.serve_forever, daemon=True,
                         name='krypto-cockpit-v63-server').start()
        return server
    raise OSError('Kein freier Port für die Cockpit-Browseransicht.')


# Open the cockpit in its own browser tab when the cockpit cell runs.
# False: only show the link.
AUTOMATISCH_OEFFNEN = True
LINK_TEXT = 'Cockpit im Browser öffnen'
TAB_NAME = 'krypto-cockpit-v63'   # re-running the cell reuses this tab instead of opening another

POPUP_HINWEIS = ('Pop-up blockiert: Der Browser hat das automatische Öffnen verhindert. '
                 'Einmalig Pop-ups für colab.research.google.com erlauben (Symbol rechts in der '
                 'Adressleiste), dann öffnet sich das Cockpit bei jedem Start von selbst. '
                 'Bis dahin den Link oben anklicken.')
GEOEFFNET_HINWEIS = 'Cockpit wurde in einem eigenen Browser-Tab geöffnet.'

_BROWSER_VIEW_JS = """(async (port, path, linkText, tabName, automatisch, blockiert, geoeffnet) => {
  const element = window.element || document.body;
  const box = document.createElement('div');
  box.style.cssText = 'font:14px system-ui,sans-serif;margin:6px 0';
  box.textContent = 'Cockpit-Link wird ermittelt …';
  element.appendChild(box);
  let url = null;
  try {
    url = new URL(path, await google.colab.kernel.proxyPort(port, {cache: false})).toString();
  } catch (e) {
    box.textContent = 'Cockpit-Link nicht verfügbar (' + e + '). Die Cockpit-Zelle erneut ausführen.';
    return;
  }
  box.textContent = '';
  const link = document.createElement('a');   // fallback: always shown
  link.href = url;
  link.target = tabName;
  link.textContent = linkText;
  box.appendChild(link);
  if (!automatisch) { return; }
  let tab = null;
  try { tab = window.open(url, tabName); } catch (e) { tab = null; }
  const note = document.createElement('div');
  note.style.cssText = 'margin-top:6px;padding:8px 10px;border-radius:6px';
  if (tab) {
    note.style.cssText += ';color:#1f6f3a;background:#eaf7ee;border:1px solid #9ad3ad';
    note.textContent = geoeffnet;
  } else {
    note.style.cssText += ';color:#7a4b00;background:#fff4e5;border:2px solid #f0a020;font-weight:600';
    note.textContent = blockiert;
  }
  box.appendChild(note);
})(%s)"""


def browser_view_js(port, path='/', automatisch=AUTOMATISCH_OEFFNEN):
    """JavaScript for the Colab cell output: always the link, and if `automatisch`
    also window.open with a clear hint when the browser blocks the pop-up."""
    args = [port, path, LINK_TEXT, TAB_NAME, bool(automatisch), POPUP_HINWEIS, GEOEFFNET_HINWEIS]
    return _BROWSER_VIEW_JS % ', '.join(json.dumps(a, ensure_ascii=False) for a in args)


_OPEN_TAB_JS = """((url, tabName, blockiert, geoeffnet) => {
  const element = window.element || document.body;
  let tab = null;
  try { tab = window.open(url, tabName); } catch (e) { tab = null; }
  const note = document.createElement('div');
  note.style.cssText = 'font:14px system-ui,sans-serif;margin-top:6px;padding:8px 10px;border-radius:6px';
  if (tab) {
    note.style.cssText += ';color:#1f6f3a;background:#eaf7ee;border:1px solid #9ad3ad';
    note.textContent = geoeffnet;
  } else {
    note.style.cssText += ';color:#7a4b00;background:#fff4e5;border:2px solid #f0a020;font-weight:600';
    note.textContent = blockiert;
  }
  element.appendChild(note);
})(%s)"""


def open_tab_js(url):
    """Try to open the known cockpit URL in its own tab and say whether it worked."""
    args = [url, TAB_NAME, POPUP_HINWEIS, GEOEFFNET_HINWEIS]
    return _OPEN_TAB_JS % ', '.join(json.dumps(a, ensure_ascii=False) for a in args)


def link_html(url):
    """The cockpit link as plain HTML: visible even if JavaScript in the output fails."""
    return (f'<div style="font:15px system-ui,sans-serif;margin:8px 0">'
            f'<a href="{safe(url)}" target="{safe(TAB_NAME)}" style="display:inline-block;padding:9px 16px;'
            f'border-radius:8px;background:#1d4ed8;color:white;font-weight:600;text-decoration:none">'
            f'{safe(LINK_TEXT)}</a><br><small style="color:#58677a">{safe(url)}</small></div>')


def colab_proxy_url(port, timeout=30):
    """Ask the Colab page for the proxied URL of `port` (Python side, like Colab's own docs)."""
    from google.colab.output import eval_js
    url = eval_js(f'google.colab.kernel.proxyPort({int(port)}, {{"cache": false}})', timeout_sec=timeout)
    if not isinstance(url, str) or not url.startswith('http'):
        raise RuntimeError(f'unerwartete Antwort: {url!r}')
    return url


def open_browser_view(cockpit, port=8771, automatisch=AUTOMATISCH_OEFFNEN):
    """Start the server and show the link; with `automatisch` also open the cockpit in its own tab."""
    server = serve_cockpit(cockpit, port)
    port = server.server_address[1]
    try:
        from google.colab import output   # noqa: F401  (only to detect Colab)
    except ImportError:
        url = f'http://localhost:{port}/'
        print(f'Cockpit im Browser: {url}')
        if automatisch:
            import webbrowser
            webbrowser.open(url, new=2)
        return server
    from IPython.display import HTML, Javascript, display
    try:
        url = colab_proxy_url(port)
    except Exception as exc:
        # Fallback: let the page's JavaScript fetch the link (V6.1 approach)
        print('Cockpit-Link nicht direkt ermittelt (' + type(exc).__name__ + ') - Ersatzweg:')
        display(Javascript(browser_view_js(port, '/', automatisch)))
    else:
        cockpit.link = url
        display(HTML(link_html(url)))
        print(f'Cockpit-Link: {url}')
        if automatisch:
            display(Javascript(open_tab_js(url)))
    print('Der Link funktioniert, solange diese Colab-Sitzung läuft; '
          f'die Seite lädt sich alle {BROWSER_REFRESH_SECONDS} s selbst neu.')
    return server


## Cockpit

Zuerst diese Zelle ausführen: Sie zeigt das Cockpit, darunter einen blauen Knopf „Cockpit im Browser öffnen“ mit dem Link, und versucht, das Cockpit selbst in einem eigenen Tab zu öffnen (blockiert der Browser das, erscheint ein orangefarbener Hinweis; dann einmal den Knopf klicken oder Pop-ups für Colab erlauben). Es wird von der Startzelle darunter laufend aktualisiert. Für eine Vollbildansicht ohne Code und Textlog im Ausgabemenü dieser Zelle „Copy link to fullscreen output“ wählen.

In [ ]:
import importlib
import sys

if '/content' not in sys.path:
    sys.path.insert(0, '/content')

import crypto_bot_v63 as bot
bot = importlib.reload(bot)
import crypto_cockpit_v63 as cockpit_ui
cockpit_ui = importlib.reload(cockpit_ui)

cockpit = cockpit_ui.Cockpit(display_enabled=True,
                             latest_path=bot.STANDARD_ORDNER / 'cockpit_v63_latest.html')

# Die Ansicht gehört zu DIESER Zelle und wird aus der Startzelle
# darunter aktualisiert. Hier steht also nur das Cockpit, ohne Textlog:
# Ausgabemenü dieser Zelle -> "Copy link to fullscreen output".
cockpit.show_placeholder()

# True: Das Cockpit öffnet sich beim Start in einem eigenen Browser-Tab.
# Beim ersten Mal Pop-ups für colab.research.google.com erlauben.
# False: nur der Link.
COCKPIT_AUTOMATISCH_OEFFNEN = True

try:   # Browser-Tab zusätzlich; liefert nur die fertige Ansicht aus
    cockpit_ui.open_browser_view(cockpit, automatisch=COCKPIT_AUTOMATISCH_OEFFNEN)
except Exception as exc:
    print('Browser-Link nicht verfügbar:', type(exc).__name__,
          '– stattdessen MyDrive/Trading_Bot/cockpit_v63_latest.html öffnen.')

## Start

Prüft zuerst die Alpaca-Paper-Schlüssel aus den Colab-Secrets (nur lesend). Fehlt ein Secret, ist es nicht freigegeben oder ungültig, erscheint hier und im Cockpit ein roter Hinweis mit der Lösung, und der Bot startet nicht. Sonst startet die Live-Demo. Stoppen über ■ (Ausführung unterbrechen); das virtuelle Konto bleibt in Google Drive gespeichert.

In [ ]:
import sys
if '/content' not in sys.path:
    sys.path.insert(0, '/content')

from google.colab import userdata
from alpaca.data.historical import CryptoHistoricalDataClient
from alpaca.trading.client import TradingClient
import crypto_bot_v63 as bot
import crypto_cockpit_v63 as cockpit_ui

cockpit = globals().get('cockpit')   # None, falls die Cockpit-Zelle nicht lief
if cockpit is None:
    print('Hinweis: Die Cockpit-Zelle wurde nicht ausgeführt - der Bot läuft ohne Cockpit.')

# Schlüssel lesen und mit je einer lesenden Abfrage bei Alpaca prüfen.
# Fehlt ein Secret, ist es nicht freigegeben oder lehnt Alpaca die Schlüssel ab,
# erscheint hier und im Cockpit ein roter Hinweis, und der Bot startet nicht.
# Der Bot enthält keine Order-Funktion; das Paper-Konto wird nicht verändert.
try:
    paper_key, paper_secret = bot.secrets_lesen(userdata.get)
    broker = TradingClient(api_key=paper_key, secret_key=paper_secret,
                           paper=True, url_override='https://paper-api.alpaca.markets')
    datenquelle = CryptoHistoricalDataClient(api_key=paper_key, secret_key=paper_secret)
    del paper_key, paper_secret
    bot.alpaca_zugang_pruefen(broker, datenquelle)
except bot.ZugangsFehler as fehler:
    cockpit_ui.startfehler_anzeigen(fehler, cockpit)
else:
    # Die farbige Textausgabe erscheint hier, das Cockpit oben.
    bot.start(datenquelle, broker=broker, cockpit=cockpit)